# Bộ bài toán ML cho panel tài chính quý (`ticker × quarter`) — bản complete-quarter + target-quarter split + RAG export

Notebook này là **experimental suite trong một file duy nhất**. Mục tiêu là kiểm tra nhiều bài toán và nhiều họ model trước khi tách code sang `.py`.

## Phạm vi dữ liệu hiện tại

- Dữ liệu dự kiến: **2024Q1 đến 2026Q1**.
- Đơn vị quan sát: một công ty trong một quý.
- Supervised learning: dùng thông tin đã công bố ở quý `t` để dự đoán quý `t+1`.
- Với khoảng 9 quý, trọng tâm là panel tabular learning; không dùng LSTM/Transformer/VAR riêng từng mã làm model chính.

## Các sửa đổi quan trọng trong bản này

1. **Không join macro annual vào panel quý**. Macro chỉ được dùng nếu có `release_date` hợp lệ và được tổng hợp point-in-time.
2. Thêm `information_cutoff` từ `disclosures`; các metric chính thức chỉ dùng dòng có thời điểm công bố.
3. Kiểm tra và xử lý duplicate line item theo phiên bản; không dùng `aggfunc="first"` tùy ý.
4. Tách preprocessing:
   - Linear/SVM/MLP: median imputation + `RobustScaler`.
   - Tree/boosting: median imputation, không bắt buộc scale.
5. Tự loại feature toàn `NaN` trong từng train fold.
6. Sửa directional accuracy thành hướng thay đổi so với giá trị hiện tại.
7. Khóa final test bằng `RUN_FINAL_TEST=False`.
8. Ranking dùng `qid` theo từng quý và relevance grade không âm.
9. Causal mặc định tắt; chỉ dùng outcome quý sau và đúng loại estimator.
10. Clustering transition dùng một model K-Means cố định để nhãn cluster có nghĩa nhất quán.
11. Anomaly detector fit trên lịch sử và score quý mới nhất.
12. Phân biệt rõ kết quả exploratory với kết quả đủ điều kiện dùng trong luận văn.

## Sáu nhóm bài toán

```text
A. Supervised regression
B. Supervised classification
C. Learning to rank
D. Unsupervised clustering
E. Unsupervised anomaly detection
F. Causal / relational analysis
```

> Kết quả trong notebook chỉ được đưa vào luận văn sau khi kiểm tra data lineage, flow-value convention, point-in-time cutoff và khóa split/model version.

## Bản runtime-safe

Bản này bổ sung **runtime profile, tiến độ từng model/fold, ETA động, heartbeat khi fit lâu và soft budget**. Mặc định `RUNTIME_PROFILE="balanced"` không chạy SVM/MLP chậm. Muốn khảo sát toàn bộ model, đổi sang `"full"`; muốn kiểm tra pipeline nhanh, dùng `"fast"`.


> Bản này loại các quý chưa đủ coverage, giữ final test target tại 2026Q1, và xuất prediction/model card có cấu trúc cho RAG.

In [21]:
import sys
import warnings
import time
import threading
from collections import defaultdict
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "src").exists(), (
    "Chạy notebook từ thư mục gốc project, nơi có folder src/."
)
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 100)
np.random.seed(42)

from src.core.db import engine, check_connection

check_connection()

ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "ml_task_suite"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

RUN_FINAL_TEST = False

# ------------------------------------------------------------------
# Official time boundary / quarter coverage
# quarter_index = fiscal_year * 4 + quarter_num.
# 2026Q1 = 2026*4+1 = 8105.
# ------------------------------------------------------------------
ANALYSIS_CUTOFF_QUARTER = 2026 * 4 + 1
CURRENT_ANALYSIS_QUARTER = ANALYSIS_CUTOFF_QUARTER
FINAL_TEST_TARGET_QUARTER = ANALYSIS_CUTOFF_QUARTER
MIN_COMPLETE_QUARTER_RATIO = 0.80
MIN_COMPLETE_QUARTER_TICKERS = 100

# Flow-derived tasks are skipped until ETL convention is confirmed.
ALLOW_UNCONFIRMED_FLOW_TASKS = False

# Structured output for downstream RAG / structured retrieval.
EXPORT_RAG_OUTPUTS = False
MODELS_LOCKED_FOR_RAG = False
WRITE_RAG_OUTPUT_TO_DB = False
RAG_MODEL_VERSION_PREFIX = "quarterly_panel_v2"
CLASSIFICATION_THRESHOLDS = {
    "margin_up_next_q": 0.50,
    "margin_negative_next_q": 0.50,
    "ocf_negative_next_q": 0.50,
    "deterioration_strict_3of3": 0.50,
    "deterioration_3of4": 0.50,
}
# Ví dụ sau khi chốt: {'net_margin_next_q': 'elastic_net'}
FINAL_MODEL_SELECTION = {}
REQUIRE_POINT_IN_TIME_CUTOFF = True
USE_SAFE_QUARTERLY_MACRO = False

# Xác nhận từ nguồn ETL trước khi đổi sang "ytd_cumulative".
# [2026-08-04] Xác nhận qua đối chiếu Q1+Q2 vs "6 tháng" trên ABI/BSG/CRC/CKG
# trong data/raw/vietstock_exports/ (xem docs/CHECKLIST_PHASES.md muc 0.1):
# quy la standalone, khong phai YTD cumulative.
FLOW_VALUE_MODE = "reported_quarter"
FLOW_MODE_CONFIRMED = True

RUN_CAUSAL = False

# ------------------------------------------------------------------
# Runtime controls
# ------------------------------------------------------------------
# fast: model đại diện, phù hợp smoke test.
# balanced: leaderboard chính, bỏ SVM/MLP chậm. Đây là mặc định.
# full: chạy tất cả model, có thể lâu đáng kể.
RUNTIME_PROFILE = "balanced"
SHOW_MODEL_PROGRESS = True
HEARTBEAT_SECONDS = 20
MODEL_SOFT_BUDGET_SECONDS = 5 * 60
TASK_SOFT_BUDGET_SECONDS = 20 * 60
SKIP_REMAINING_FOLDS_AFTER_BUDGET = True

# SVM chỉ xuất hiện trong profile full và vẫn bị cap sample.
SLOW_MODEL_MAX_TRAIN_N = 500
SVM_MAX_ITER = 5_000

MODEL_PROFILES = {
    "fast": {
        "regression": [
            "ridge",
            "elastic_net",
            "random_forest",
            "hist_gbm",
            "xgboost",
        ],
        "classification": [
            "logistic",
            "elastic_net_logistic",
            "random_forest",
            "hist_gbm",
            "xgboost",
        ],
    },
    "balanced": {
        "regression": [
            "linear_regression",
            "ridge",
            "lasso",
            "elastic_net",
            "huber",
            "knn",
            "random_forest",
            "extra_trees",
            "hist_gbm",
            "xgboost",
            "lightgbm",
            "catboost",
        ],
        "classification": [
            "logistic",
            "elastic_net_logistic",
            "knn",
            "random_forest",
            "extra_trees",
            "hist_gbm",
            "xgboost",
            "lightgbm",
            "catboost",
        ],
    },
    "full": {
        "regression": None,
        "classification": None,
    },
}

if RUNTIME_PROFILE not in MODEL_PROFILES:
    raise ValueError(
        f"RUNTIME_PROFILE phải là một trong {list(MODEL_PROFILES)}"
    )

# Giảm số cây/iterations ở fast/balanced; full giữ cấu hình đầy đủ.
TREE_ESTIMATORS = {"fast": 120, "balanced": 220, "full": 400}[RUNTIME_PROFILE]
BOOST_ITERATIONS = {"fast": 150, "balanced": 250, "full": 400}[RUNTIME_PROFILE]
MLP_MAX_ITER = {"fast": 400, "balanced": 700, "full": 1500}[RUNTIME_PROFILE]

print("PROJECT_ROOT:", PROJECT_ROOT)
print("ARTIFACT_DIR:", ARTIFACT_DIR)
print("RUN_FINAL_TEST:", RUN_FINAL_TEST)
print("ANALYSIS_CUTOFF_QUARTER:", ANALYSIS_CUTOFF_QUARTER)
print("FINAL_TEST_TARGET_QUARTER:", FINAL_TEST_TARGET_QUARTER)
print("MIN_COMPLETE_QUARTER_RATIO:", MIN_COMPLETE_QUARTER_RATIO)
print("ALLOW_UNCONFIRMED_FLOW_TASKS:", ALLOW_UNCONFIRMED_FLOW_TASKS)
print("EXPORT_RAG_OUTPUTS:", EXPORT_RAG_OUTPUTS)
print("REQUIRE_POINT_IN_TIME_CUTOFF:", REQUIRE_POINT_IN_TIME_CUTOFF)
print("USE_SAFE_QUARTERLY_MACRO:", USE_SAFE_QUARTERLY_MACRO)
print("FLOW_VALUE_MODE:", FLOW_VALUE_MODE, "| confirmed:", FLOW_MODE_CONFIRMED)
print("RUNTIME_PROFILE:", RUNTIME_PROFILE)
print("TREE_ESTIMATORS:", TREE_ESTIMATORS, "| BOOST_ITERATIONS:", BOOST_ITERATIONS)
print("HEARTBEAT_SECONDS:", HEARTBEAT_SECONDS)
print("MODEL_SOFT_BUDGET_SECONDS:", MODEL_SOFT_BUDGET_SECONDS)
print("TASK_SOFT_BUDGET_SECONDS:", TASK_SOFT_BUDGET_SECONDS)


def _try_import(name):
    try:
        return __import__(name)
    except Exception as exc:
        print(
            f"[optional] {name}: chưa dùng được "
            f"({type(exc).__name__}: {exc}); phần phụ thuộc package này sẽ bị bỏ qua."
        )
        return None


HAS_LIGHTGBM = _try_import("lightgbm") is not None
HAS_CATBOOST = _try_import("catboost") is not None
HAS_UMAP = _try_import("umap") is not None

print(
    f"lightgbm: {'có' if HAS_LIGHTGBM else 'không'} | "
    f"catboost: {'có' if HAS_CATBOOST else 'không'} | "
    f"umap: {'có' if HAS_UMAP else 'không'}"
)

OK - connected to localhost:5432/financial_db
PROJECT_ROOT: /Users/mac/Desktop/Project/Thesis_Project
ARTIFACT_DIR: /Users/mac/Desktop/Project/Thesis_Project/artifacts/ml_task_suite
RUN_FINAL_TEST: False
ANALYSIS_CUTOFF_QUARTER: 8105
FINAL_TEST_TARGET_QUARTER: 8105
MIN_COMPLETE_QUARTER_RATIO: 0.8
ALLOW_UNCONFIRMED_FLOW_TASKS: False
EXPORT_RAG_OUTPUTS: False
REQUIRE_POINT_IN_TIME_CUTOFF: True
USE_SAFE_QUARTERLY_MACRO: False
FLOW_VALUE_MODE: reported_quarter | confirmed: True
RUNTIME_PROFILE: balanced
TREE_ESTIMATORS: 220 | BOOST_ITERATIONS: 250
HEARTBEAT_SECONDS: 20
MODEL_SOFT_BUDGET_SECONDS: 300
TASK_SOFT_BUDGET_SECONDS: 1200
[optional] umap: chưa dùng được (ModuleNotFoundError: No module named 'umap'); phần phụ thuộc package này sẽ bị bỏ qua.
lightgbm: có | catboost: có | umap: không


## 1. Xây panel theo quý (`ticker × quarter`)

Notebook tái sử dụng `_fetch_raw`, `_PANEL_FIELDS` và `_RATIO_BOUNDS` từ project, nhưng có các thay đổi:

- chỉ giữ `period ∈ {Q1,Q2,Q3,Q4}`;
- giải quyết duplicate trước pivot;
- ưu tiên consolidated;
- gắn `information_cutoff` từ `disclosures`;
- không join macro annual;
- ratio được chia an toàn để tránh mẫu số bằng 0;
- có audit cho flow values và restatement risk.

### Cảnh báo về flow values

Các biến như `revenue`, `net_profit`, `operating_cash_flow` có thể được lưu theo:

- giá trị riêng từng quý; hoặc
- lũy kế từ đầu năm.

Phải xác nhận convention trước khi dùng tăng trưởng QoQ. Bản notebook mặc định giữ nguyên dữ liệu (`reported_quarter`) nhưng đánh dấu `FLOW_MODE_CONFIRMED=False`.

In [22]:
from src.causal.panel_builder import _fetch_raw, _PANEL_FIELDS, _RATIO_BOUNDS
from src.core.constants import FINANCE_SUBSECTOR_KEYWORDS
from sqlalchemy import text

QUARTER_PERIODS = ["Q1", "Q2", "Q3", "Q4"]
RAW_COLS = list(_PANEL_FIELDS.values())
RATIO_COLS = [
    "leverage",
    "current_ratio",
    "gross_margin",
    "net_margin",
    "asset_turnover",
    "ocf_to_revenue",
]

FLOW_COLS = [
    "revenue",
    "cogs",
    "gross_profit",
    "profit_before_tax",
    "net_profit",
    "operating_cash_flow",
]

CORE_COLS = RAW_COLS + RATIO_COLS


def _table_columns(table_name: str) -> set[str]:
    query = text('''
    SELECT column_name
    FROM information_schema.columns
    WHERE table_schema = 'public'
      AND table_name = :table_name
    ''')
    return set(
        pd.read_sql(query, engine, params={"table_name": table_name})[
            "column_name"
        ].tolist()
    )


def _safe_divide(
    numerator: pd.Series,
    denominator: pd.Series,
    eps: float = 1e-12,
) -> pd.Series:
    numerator = pd.to_numeric(numerator, errors="coerce")
    denominator = pd.to_numeric(denominator, errors="coerce")
    result = numerator / denominator.where(denominator.abs() > eps)
    return result.replace([np.inf, -np.inf], np.nan)


def _normalise_consolidation(value) -> str:
    text = str(value).strip().lower()
    if text in {"consolidated", "hop_nhat", "hợp nhất", "1", "true"}:
        return "consolidated"
    if text in {"standalone", "separate", "rieng", "riêng", "0", "false"}:
        return "standalone"
    return text or "unknown"


def _resolve_duplicate_line_items(qraw: pd.DataFrame) -> pd.DataFrame:
    '''
    Không pivot với aggfunc='first' khi có nhiều phiên bản cùng key.
    Duplicate cùng value được gộp; duplicate khác value cần version metadata.
    '''
    key = [
        "ticker",
        "fiscal_year",
        "period",
        "consolidation",
        "statement_type",
        "ms_code",
    ]
    key = [col for col in key if col in qraw.columns]

    duplicated = qraw.duplicated(key, keep=False)
    if not duplicated.any():
        return qraw

    dup = qraw.loc[duplicated].copy()
    value_nunique = dup.groupby(key, dropna=False)["value"].nunique(dropna=False)
    ambiguous_keys = value_nunique[value_nunique > 1]

    print(
        f"[duplicate audit] {duplicated.sum()} dòng duplicate; "
        f"{len(ambiguous_keys)} key có nhiều value khác nhau."
    )

    exact = qraw.drop_duplicates(key + ["value"], keep="last")

    if len(ambiguous_keys) == 0:
        return exact

    version_candidates = [
        col
        for col in [
            "disclosure_date",
            "signed_date",
            "updated_at",
            "ingested_at",
            "created_at",
            "id",
        ]
        if col in exact.columns
    ]

    if not version_candidates:
        raise RuntimeError(
            "Có nhiều giá trị khác nhau cho cùng line-item key nhưng _fetch_raw() "
            "không trả version timestamp/id. Không thể chọn canonical row an toàn. "
            "Hãy bổ sung source/version metadata vào _fetch_raw()."
        )

    sortable = exact.copy()
    for col in version_candidates:
        if col != "id":
            sortable[col] = pd.to_datetime(sortable[col], errors="coerce")

    sortable = sortable.sort_values(key + version_candidates)
    resolved = sortable.drop_duplicates(key, keep="last")

    print(
        "[duplicate audit] Đã chọn canonical row theo: "
        + ", ".join(version_candidates)
    )
    return resolved


def _load_disclosure_cutoffs() -> pd.DataFrame:
    cols = _table_columns("disclosures")
    required = {"ticker", "fiscal_year", "period"}
    if not required.issubset(cols):
        print(
            "[point-in-time] disclosures thiếu ticker/fiscal_year/period; "
            "không thể gắn information_cutoff."
        )
        return pd.DataFrame()

    selected = ["ticker", "fiscal_year", "period"]
    selected.append(
        "consolidation" if "consolidation" in cols
        else "'unknown' AS consolidation"
    )
    selected.append(
        "disclosure_date" if "disclosure_date" in cols
        else "NULL AS disclosure_date"
    )
    selected.append(
        "signed_date" if "signed_date" in cols
        else "NULL AS signed_date"
    )

    disclosures = pd.read_sql(
        f"SELECT {', '.join(selected)} FROM disclosures",
        engine,
    )
    disclosures["period"] = disclosures["period"].astype(str).str.upper()
    disclosures["consolidation"] = disclosures["consolidation"].map(
        _normalise_consolidation
    )
    disclosures["disclosure_date"] = pd.to_datetime(
        disclosures["disclosure_date"], errors="coerce"
    )
    disclosures["signed_date"] = pd.to_datetime(
        disclosures["signed_date"], errors="coerce"
    )

    disclosures["information_cutoff"] = disclosures[
        "disclosure_date"
    ].combine_first(disclosures["signed_date"])

    disclosures = disclosures[
        disclosures["information_cutoff"].notna()
    ].copy()

    if disclosures.empty:
        return disclosures

    exact = (
        disclosures.groupby(
            ["ticker", "fiscal_year", "period", "consolidation"],
            as_index=False,
        )["information_cutoff"]
        .min()
    )

    fallback = (
        disclosures.groupby(
            ["ticker", "fiscal_year", "period"],
            as_index=False,
        )["information_cutoff"]
        .min()
        .rename(columns={"information_cutoff": "information_cutoff_fallback"})
    )

    return exact.merge(
        fallback,
        on=["ticker", "fiscal_year", "period"],
        how="left",
    )


def _attach_information_cutoff(wide: pd.DataFrame) -> pd.DataFrame:
    cutoffs = _load_disclosure_cutoffs()
    output = wide.copy()

    if cutoffs.empty:
        output["information_cutoff"] = pd.NaT
        output["cutoff_match_quality"] = "missing"
        return output

    output["consolidation"] = output["consolidation"].map(
        _normalise_consolidation
    )

    exact_cols = [
        "ticker",
        "fiscal_year",
        "period",
        "consolidation",
        "information_cutoff",
    ]
    fallback = cutoffs[
        [
            "ticker",
            "fiscal_year",
            "period",
            "information_cutoff_fallback",
        ]
    ].drop_duplicates()

    output = output.merge(
        cutoffs[exact_cols],
        on=["ticker", "fiscal_year", "period", "consolidation"],
        how="left",
    )
    output = output.merge(
        fallback,
        on=["ticker", "fiscal_year", "period"],
        how="left",
    )

    exact_match = output["information_cutoff"].notna()
    output["information_cutoff"] = output[
        "information_cutoff"
    ].combine_first(output["information_cutoff_fallback"])
    output["cutoff_match_quality"] = np.where(
        exact_match,
        "exact",
        np.where(
            output["information_cutoff"].notna(),
            "fallback_without_consolidation",
            "missing",
        ),
    )
    return output.drop(columns=["information_cutoff_fallback"])


def _convert_ytd_to_standalone(
    wide: pd.DataFrame,
    flow_cols: list[str],
) -> pd.DataFrame:
    output = wide.sort_values(
        ["ticker", "fiscal_year", "quarter_num"]
    ).copy()
    group = output.groupby(
        ["ticker", "fiscal_year", "consolidation"],
        group_keys=False,
    )

    for col in flow_cols:
        if col not in output.columns:
            continue
        previous = group[col].shift(1)
        previous_q = group["quarter_num"].shift(1)
        consecutive = output["quarter_num"] - previous_q == 1
        standalone = output[col] - previous
        output[col] = np.where(
            output["quarter_num"].eq(1),
            output[col],
            np.where(consecutive, standalone, np.nan),
        )
    return output


def audit_flow_value_convention(
    wide: pd.DataFrame,
    flow_cols: list[str] = FLOW_COLS,
) -> pd.DataFrame:
    rows = []
    ordered = wide.sort_values(
        ["ticker", "fiscal_year", "quarter_num"]
    )
    group = ordered.groupby(
        ["ticker", "fiscal_year", "consolidation"],
        group_keys=False,
    )

    for col in flow_cols:
        if col not in ordered.columns:
            continue
        previous = group[col].shift(1)
        valid = ordered[col].notna() & previous.notna()
        if valid.sum() == 0:
            continue
        ratio = _safe_divide(
            ordered.loc[valid, col],
            previous.loc[valid],
        )
        rows.append(
            {
                "feature": col,
                "n_transitions": int(valid.sum()),
                "share_non_decreasing": float(
                    (ordered.loc[valid, col] >= previous.loc[valid]).mean()
                ),
                "median_current_to_previous": float(np.nanmedian(ratio)),
            }
        )
    return pd.DataFrame(rows)


def _attach_safe_quarterly_macro(wide: pd.DataFrame) -> pd.DataFrame:
    '''
    Conservative point-in-time macro join.
    Chỉ dùng macro Q1..Q4 có release_date <= information_cutoff.
    '''
    output = wide.copy()

    macro_cols = _table_columns("macro_indicators")
    required = {
        "indicator_name",
        "value",
        "fiscal_year",
        "period",
        "release_date",
    }
    if not required.issubset(macro_cols):
        print(
            "[macro] Thiếu quarterly release-dated macro fields; bỏ qua macro."
        )
        return output

    macro = pd.read_sql(
        '''
        SELECT indicator_name, value, fiscal_year, period, release_date
        FROM macro_indicators
        WHERE release_date IS NOT NULL
        ''',
        engine,
    )
    macro["period"] = macro["period"].astype(str).str.upper()
    macro = macro[macro["period"].isin(QUARTER_PERIODS)].copy()
    macro["release_date"] = pd.to_datetime(
        macro["release_date"], errors="coerce"
    )
    macro["value"] = pd.to_numeric(macro["value"], errors="coerce")
    macro = macro.dropna(subset=["release_date", "value"])

    if macro.empty:
        print("[macro] Không có macro quarterly với release_date; bỏ qua.")
        return output

    rows = []
    for row in output[
        ["ticker", "fiscal_year", "period", "information_cutoff"]
    ].itertuples(index=False):
        eligible = macro[
            (macro["fiscal_year"] == row.fiscal_year)
            & (macro["period"] == row.period)
            & (macro["release_date"] <= row.information_cutoff)
        ]
        record = {
            "ticker": row.ticker,
            "fiscal_year": row.fiscal_year,
            "period": row.period,
        }
        for name, group in eligible.groupby("indicator_name"):
            record[f"macro_{name}"] = group.sort_values(
                "release_date"
            )["value"].iloc[-1]
        rows.append(record)

    macro_wide = pd.DataFrame(rows)
    if macro_wide.empty:
        return output

    return output.merge(
        macro_wide,
        on=["ticker", "fiscal_year", "period"],
        how="left",
    )


def build_quarterly_panel() -> pd.DataFrame:
    raw = _fetch_raw()
    if raw.empty:
        print(
            "[cảnh báo] financial_line_items rỗng; không thể xây panel quý."
        )
        return pd.DataFrame()

    qraw = raw[
        raw["period"].astype(str).str.upper().isin(QUARTER_PERIODS)
    ].copy()
    if qraw.empty:
        print(
            "[cảnh báo] Không có Q1..Q4. Kiểm tra dữ liệu ingestion."
        )
        return pd.DataFrame()

    qraw["period"] = qraw["period"].astype(str).str.upper()
    qraw["quarter_num"] = qraw["period"].str[1].astype(int)
    qraw["consolidation"] = qraw["consolidation"].map(
        _normalise_consolidation
    )
    qraw["field"] = list(
        zip(qraw["statement_type"], qraw["ms_code"])
    )
    qraw["field"] = qraw["field"].map(_PANEL_FIELDS)
    qraw["value"] = pd.to_numeric(qraw["value"], errors="coerce")
    qraw = qraw[qraw["field"].notna()].copy()

    qraw = _resolve_duplicate_line_items(qraw)

    idx_cols = [
        "ticker",
        "fiscal_year",
        "period",
        "quarter_num",
        "consolidation",
        "sector",
        "exchange",
        "company_name",
    ]

    wide = (
        qraw.pivot(
            index=idx_cols,
            columns="field",
            values="value",
        )
        .reset_index()
    )
    wide.columns.name = None

    for col in _PANEL_FIELDS.values():
        if col not in wide.columns:
            wide[col] = np.nan

    wide["_cons_rank"] = np.select(
        [
            wide["consolidation"].eq("consolidated"),
            wide["consolidation"].eq("standalone"),
        ],
        [0, 1],
        default=2,
    )
    wide = (
        wide.sort_values(
            [
                "ticker",
                "fiscal_year",
                "quarter_num",
                "_cons_rank",
            ]
        )
        .drop_duplicates(
            subset=["ticker", "fiscal_year", "quarter_num"],
            keep="first",
        )
        .drop(columns=["_cons_rank"])
        .reset_index(drop=True)
    )

    flow_audit = audit_flow_value_convention(wide)
    if not flow_audit.empty:
        print("[flow audit] Dùng bảng sau để xác nhận cumulative/standalone:")
        display(flow_audit)

    if not FLOW_MODE_CONFIRMED:
        print(
            "[CẢNH BÁO] FLOW_MODE_CONFIRMED=False. "
            "Kết quả liên quan QoQ chỉ mang tính exploratory."
        )

    if FLOW_VALUE_MODE == "ytd_cumulative":
        wide = _convert_ytd_to_standalone(wide, FLOW_COLS)
    elif FLOW_VALUE_MODE != "reported_quarter":
        raise ValueError(
            "FLOW_VALUE_MODE phải là 'reported_quarter' hoặc 'ytd_cumulative'."
        )

    wide["leverage"] = _safe_divide(
        wide["total_liabilities"], wide["total_assets"]
    )
    wide["current_ratio"] = _safe_divide(
        wide["current_assets"], wide["current_liabilities"]
    )
    wide["gross_margin"] = _safe_divide(
        wide["gross_profit"], wide["revenue"]
    )
    wide["net_margin"] = _safe_divide(
        wide["net_profit"], wide["revenue"]
    )
    wide["asset_turnover"] = _safe_divide(
        wide["revenue"], wide["total_assets"]
    )
    wide["ocf_to_revenue"] = _safe_divide(
        wide["operating_cash_flow"], wide["revenue"]
    )

    is_finance = wide["company_name"].str.contains(
        "|".join(FINANCE_SUBSECTOR_KEYWORDS),
        case=False,
        na=False,
    )

    def _violations(row):
        return [
            col
            for col, (low, high) in _RATIO_BOUNDS.items()
            if pd.notna(row[col]) and not (low <= row[col] <= high)
        ]

    violations = wide.apply(_violations, axis=1)
    eligible = ~is_finance & violations.map(len).eq(0)

    print(
        f"[quarterly_panel] {len(wide)} dòng thô -> "
        f"loại {int(is_finance.sum())} finance-subsector, "
        f"{int((~is_finance & violations.map(len).gt(0)).sum())} ratio-invalid "
        f"-> {int(eligible.sum())} dòng."
    )

    wide = wide[eligible].copy()
    wide = _attach_information_cutoff(wide)

    if REQUIRE_POINT_IN_TIME_CUTOFF:
        before = len(wide)
        wide = wide[wide["information_cutoff"].notna()].copy()
        print(
            f"[point-in-time] Giữ {len(wide)}/{before} dòng có information_cutoff."
        )

    wide["quarter_index"] = (
        wide["fiscal_year"].astype(int) * 4
        + wide["quarter_num"].astype(int)
    )

    if USE_SAFE_QUARTERLY_MACRO:
        wide = _attach_safe_quarterly_macro(wide)
    else:
        print(
            "[macro] Macro annual/không có release_date bị loại khỏi panel quý."
        )

    wide["restatement_point_in_time_verified"] = False

    wide = (
        wide.drop(columns=["company_name"])
        .sort_values(["ticker", "quarter_index"])
        .reset_index(drop=True)
    )
    return wide


panel = build_quarterly_panel()

if not panel.empty:
    print(
        f"panel: {panel.shape[0]} dòng, "
        f"{panel['ticker'].nunique()} ticker, "
        f"{panel['quarter_index'].nunique()} quý."
    )
    display(
        panel[
            [
                "ticker",
                "fiscal_year",
                "period",
                "information_cutoff",
                "cutoff_match_quality",
                "sector",
                "net_margin",
                "leverage",
            ]
        ].head(10)
    )

[flow audit] Dùng bảng sau để xác nhận cumulative/standalone:


,feature,n_transitions,share_non_decreasing,median_current_to_previous
0,revenue,1241,0.639807,1.067934
1,cogs,1201,0.636969,1.069249
2,gross_profit,1206,0.592869,1.053485
3,profit_before_tax,1244,0.560289,0.991873
4,net_profit,1210,0.570248,1.010723
5,operating_cash_flow,41,0.463415,0.793274


[quarterly_panel] 1860 dòng thô -> loại 227 finance-subsector, 30 ratio-invalid -> 1603 dòng.
[point-in-time] Giữ 1603/1603 dòng có information_cutoff.
[macro] Macro annual/không có release_date bị loại khỏi panel quý.
panel: 1603 dòng, 177 ticker, 11 quý.


,ticker,fiscal_year,period,information_cutoff,cutoff_match_quality,sector,net_margin,leverage
0,AAA,2024,Q1,2024-04-26,exact,Nguyên vật liệu,0.048550,0.481150
1,AAA,2024,Q2,2024-07-29,exact,Nguyên vật liệu,0.039107,0.498053
2,AAA,2024,Q3,2024-11-04,exact,Nguyên vật liệu,-0.008048,0.533731
3,AAA,2024,Q4,2025-01-25,exact,Nguyên vật liệu,0.014058,0.546464
4,AAA,2025,Q1,2025-04-29,exact,Nguyên vật liệu,0.014395,0.504522
5,AAA,2025,Q2,2025-07-30,exact,Nguyên vật liệu,0.074358,0.515351
6,AAA,2025,Q3,2025-10-29,exact,Nguyên vật liệu,0.052934,0.508413
7,AAA,2025,Q4,2026-01-28,exact,Nguyên vật liệu,0.028291,0.528435
8,AAA,2026,Q1,2026-04-29,exact,Nguyên vật liệu,0.073109,0.520727
9,AAAA,2025,Q2,2026-01-13,exact,Chưa phân loại,0.038416,0.846118


## 1.1. Complete-quarter gate và official analysis cutoff

Dữ liệu ingest có thể xuất hiện sớm cho một quý mới nhưng chỉ có vài chục hoặc vài
công ty. Những quý chưa đủ coverage không được dùng làm validation, clustering,
anomaly scoring hoặc “latest quarter”.

Bản này:

- giữ dữ liệu đến **2026Q1** (`quarter_index=8105`);
- ước lượng coverage chuẩn bằng median số ticker của các quý không vượt cutoff;
- chỉ giữ quý có ít nhất `max(100, 80% × typical coverage)` ticker;
- final test được định nghĩa theo **target quarter 2026Q1**, nên feature quarter của
  final test là **2025Q4**.

In [23]:
def quarter_index_to_label(value: int | float) -> str:
    value = int(value)
    year = (value - 1) // 4
    quarter = ((value - 1) % 4) + 1
    return f"{year}Q{quarter}"


raw_panel_before_coverage_gate = panel.copy()

quarter_coverage_gate = (
    raw_panel_before_coverage_gate.groupby(
        "quarter_index",
        as_index=False,
    )
    .agg(
        rows=("ticker", "size"),
        tickers=("ticker", "nunique"),
    )
    .sort_values("quarter_index")
)

coverage_before_cutoff = quarter_coverage_gate[
    quarter_coverage_gate["quarter_index"]
    <= ANALYSIS_CUTOFF_QUARTER
].copy()

if coverage_before_cutoff.empty:
    raise RuntimeError(
        "Không có quý nào trước ANALYSIS_CUTOFF_QUARTER."
    )

typical_quarter_tickers = int(
    round(coverage_before_cutoff["tickers"].median())
)
complete_quarter_threshold = max(
    MIN_COMPLETE_QUARTER_TICKERS,
    int(np.ceil(
        typical_quarter_tickers
        * MIN_COMPLETE_QUARTER_RATIO
    )),
)

quarter_coverage_gate["quarter_label"] = (
    quarter_coverage_gate["quarter_index"]
    .map(quarter_index_to_label)
)
quarter_coverage_gate["before_cutoff"] = (
    quarter_coverage_gate["quarter_index"]
    <= ANALYSIS_CUTOFF_QUARTER
)
quarter_coverage_gate["coverage_ratio_vs_typical"] = (
    quarter_coverage_gate["tickers"]
    / max(1, typical_quarter_tickers)
)
quarter_coverage_gate["is_complete_official_quarter"] = (
    quarter_coverage_gate["before_cutoff"]
    & (
        quarter_coverage_gate["tickers"]
        >= complete_quarter_threshold
    )
)

complete_quarters = (
    quarter_coverage_gate.loc[
        quarter_coverage_gate[
            "is_complete_official_quarter"
        ],
        "quarter_index",
    ]
    .astype(int)
    .tolist()
)

if ANALYSIS_CUTOFF_QUARTER not in complete_quarters:
    raise RuntimeError(
        "Official cutoff quarter không đủ coverage. "
        f"cutoff={quarter_index_to_label(ANALYSIS_CUTOFF_QUARTER)}, "
        f"threshold={complete_quarter_threshold} tickers."
    )

excluded_quarters = quarter_coverage_gate.loc[
    ~quarter_coverage_gate[
        "is_complete_official_quarter"
    ],
    ["quarter_index", "quarter_label", "tickers"],
].copy()

panel = raw_panel_before_coverage_gate[
    raw_panel_before_coverage_gate[
        "quarter_index"
    ].isin(complete_quarters)
].copy()

CURRENT_ANALYSIS_QUARTER = int(
    panel["quarter_index"].max()
)

if CURRENT_ANALYSIS_QUARTER != ANALYSIS_CUTOFF_QUARTER:
    raise RuntimeError(
        "CURRENT_ANALYSIS_QUARTER không khớp official cutoff."
    )

print(
    "[complete-quarter gate] typical coverage=",
    typical_quarter_tickers,
    "| threshold=",
    complete_quarter_threshold,
)
print(
    "[complete-quarter gate] official quarters:",
    [
        quarter_index_to_label(value)
        for value in complete_quarters
    ],
)
print(
    "[complete-quarter gate] excluded/incomplete quarters:"
)
display(excluded_quarters)
print(
    "Official panel:",
    len(panel),
    "rows |",
    panel["ticker"].nunique(),
    "tickers | latest=",
    quarter_index_to_label(CURRENT_ANALYSIS_QUARTER),
)
print(
    "Final test:",
    quarter_index_to_label(
        FINAL_TEST_TARGET_QUARTER - 1
    ),
    "features ->",
    quarter_index_to_label(
        FINAL_TEST_TARGET_QUARTER
    ),
    "target",
)

quarter_coverage_gate.to_csv(
    ARTIFACT_DIR / "quarter_coverage_gate.csv",
    index=False,
)

[complete-quarter gate] typical coverage= 174 | threshold= 140
[complete-quarter gate] official quarters: ['2024Q1', '2024Q2', '2024Q3', '2024Q4', '2025Q1', '2025Q2', '2025Q3', '2025Q4', '2026Q1']
[complete-quarter gate] excluded/incomplete quarters:


,quarter_index,quarter_label,tickers
9,8106,2026Q2,38
10,8107,2026Q3,4


Official panel: 1561 rows | 177 tickers | latest= 2026Q1
Final test: 2025Q4 features -> 2026Q1 target


## 2. Feature engineering và labels

Các quy tắc:

- `lag1` và target chỉ được tạo khi quý thực sự liên tiếp.
- YoY chỉ được tạo khi cách đúng bốn quý.
- `target_observed_at` là ngày công bố của quý mục tiêu.
- Sector-relative features dùng median/IQR để giảm ảnh hưởng outlier.
- Có hai deterioration labels riêng:
  - `deterioration_strict_3of3`;
  - `deterioration_3of4`.

In [24]:
def _shift_valid(grouped, col, periods, valid_mask):
    return grouped[col].shift(periods).where(valid_mask)


def enrich_panel(panel: pd.DataFrame) -> pd.DataFrame:
    df = panel.sort_values(
        ["ticker", "quarter_index"]
    ).reset_index(drop=True)
    grouped = df.groupby("ticker")

    prev1_qidx = grouped["quarter_index"].shift(1)
    next1_qidx = grouped["quarter_index"].shift(-1)
    prev4_qidx = grouped["quarter_index"].shift(4)

    valid_lag1 = (df["quarter_index"] - prev1_qidx).eq(1)
    valid_next1 = (next1_qidx - df["quarter_index"]).eq(1)
    valid_lag4 = (df["quarter_index"] - prev4_qidx).eq(4)

    for col in CORE_COLS:
        if col in df.columns:
            df[f"{col}_lag1"] = _shift_valid(
                grouped, col, 1, valid_lag1
            )

    for col in [
        "net_margin",
        "gross_margin",
        "leverage",
        "current_ratio",
        "asset_turnover",
        "ocf_to_revenue",
    ]:
        if col in df.columns:
            df[f"delta_{col}"] = df[col] - df[f"{col}_lag1"]

    for base in ["revenue", "net_profit", "total_assets"]:
        if base not in df.columns:
            continue

        lag1 = df[f"{base}_lag1"]
        lag4 = _shift_valid(grouped, base, 4, valid_lag4)

        df[f"{base}_growth_qoq"] = _safe_divide(
            df[base] - lag1,
            lag1.abs(),
        )
        df[f"{base}_growth_yoy"] = _safe_divide(
            df[base] - lag4,
            lag4.abs(),
        )

    if "operating_cash_flow" in df.columns:
        df["ocf_signed_log"] = (
            np.sign(df["operating_cash_flow"])
            * np.log1p(df["operating_cash_flow"].abs())
        )

    grouped2 = df.groupby("ticker")

    next_target_cols = [
        col
        for col in (
            CORE_COLS
            + [
                "revenue_growth_qoq",
                "net_profit_growth_qoq",
                "revenue_growth_yoy",
                "ocf_signed_log",
            ]
        )
        if col in df.columns
    ]

    for col in next_target_cols:
        df[f"{col}_next_q"] = _shift_valid(
            grouped2, col, -1, valid_next1
        )

    if "information_cutoff" in df.columns:
        df["target_observed_at"] = _shift_valid(
            grouped2,
            "information_cutoff",
            -1,
            valid_next1,
        )

    df["target_quarter_index"] = (
        df["quarter_index"] + 1
    ).where(valid_next1)

    return df


def add_sector_relative_features(
    panel: pd.DataFrame,
    cols=RATIO_COLS,
) -> pd.DataFrame:
    df = panel.copy()

    for col in cols:
        grouped = df.groupby(
            ["quarter_index", "sector"]
        )[col]

        median = grouped.transform("median")
        q1 = grouped.transform(lambda s: s.quantile(0.25))
        q3 = grouped.transform(lambda s: s.quantile(0.75))
        iqr = (q3 - q1).replace(0, np.nan)

        df[f"{col}_vs_sector"] = df[col] - median
        df[f"{col}_sector_z"] = (
            (df[col] - median) / iqr
        )

    return df


def add_labels(panel: pd.DataFrame) -> pd.DataFrame:
    df = panel.copy()

    have_next_margin = df["net_margin_next_q"].notna()
    have_next_ocf = df["operating_cash_flow_next_q"].notna()
    have_next_liq = df[
        ["current_ratio_next_q", "leverage_next_q"]
    ].notna().all(axis=1)

    df["margin_up_next_q"] = np.where(
        have_next_margin,
        (df["net_margin_next_q"] > df["net_margin"]).astype(float),
        np.nan,
    )
    df["margin_negative_next_q"] = np.where(
        have_next_margin,
        (df["net_margin_next_q"] < 0).astype(float),
        np.nan,
    )
    df["ocf_negative_next_q"] = np.where(
        have_next_ocf,
        (df["operating_cash_flow_next_q"] < 0).astype(float),
        np.nan,
    )

    strict_valid = have_next_margin & have_next_liq
    strict_3of3 = (
        (df["net_margin_next_q"] < df["net_margin"])
        & (df["current_ratio_next_q"] < df["current_ratio"])
        & (df["leverage_next_q"] > df["leverage"])
    )
    df["deterioration_strict_3of3"] = np.where(
        strict_valid,
        strict_3of3.astype(float),
        np.nan,
    )

    score_valid = (
        have_next_margin
        & have_next_ocf
        & have_next_liq
    )
    score = (
        (df["net_margin_next_q"] < df["net_margin"]).astype(int)
        + (
            df["operating_cash_flow_next_q"]
            < df["operating_cash_flow"]
        ).astype(int)
        + (
            df["current_ratio_next_q"]
            < df["current_ratio"]
        ).astype(int)
        + (
            df["leverage_next_q"] > df["leverage"]
        ).astype(int)
    )
    df["deterioration_score"] = np.where(
        score_valid,
        score,
        np.nan,
    )
    df["deterioration_3of4"] = np.where(
        score_valid,
        (score >= 3).astype(float),
        np.nan,
    )

    df["net_margin_gain_next_q"] = (
        df["net_margin_next_q"] - df["net_margin"]
    )

    return df


if not panel.empty:
    panel = enrich_panel(panel)
    panel = add_sector_relative_features(panel)
    panel = add_labels(panel)

    label_cols = [
        "margin_up_next_q",
        "margin_negative_next_q",
        "ocf_negative_next_q",
        "deterioration_strict_3of3",
        "deterioration_3of4",
    ]
    print("Tỷ lệ positive label:")
    display(panel[label_cols].mean())

Tỷ lệ positive label:


margin_up_next_q             0.529283
margin_negative_next_q       0.116398
ocf_negative_next_q          0.511327
deterioration_strict_3of3    0.184615
deterioration_3of4           0.179153
dtype: float64

## 3. Feature groups, walk-forward split và preprocessing

### Split

- Các fold validation dùng toàn bộ quý trước đó để dự đoán quý tiếp theo.
- Quý có label mới nhất là final test và bị loại khi `RUN_FINAL_TEST=False`.
- Final test chỉ được mở sau khi đã chốt model.

### Preprocessing công bằng

- Linear, SVM, KNN và MLP: numeric median imputation + `RobustScaler`.
- Tree/boosting: numeric median imputation, không scale.
- One-hot encoder chỉ fit trên train fold.
- Feature toàn `NaN` trong train fold bị loại tự động.
- Model bị cap sample vì runtime được đánh dấu `exploratory_capped`, không đưa vào official leaderboard.

In [25]:
FEATURE_GROUPS = {
    "profitability": [
        "gross_margin",
        "net_margin",
        "profit_before_tax",
        "net_profit",
    ],
    "liquidity_leverage": [
        "leverage",
        "current_ratio",
        "current_assets",
        "current_liabilities",
        "total_liabilities",
        "total_equity",
    ],
    "efficiency": [
        "asset_turnover",
        "inventory",
        "cogs",
    ],
    "cash_quality": [
        "operating_cash_flow",
        "ocf_to_revenue",
    ],
    "growth": [
        "revenue_growth_qoq",
        "net_profit_growth_qoq",
        "total_assets_growth_qoq",
        "revenue_growth_yoy",
        "net_profit_growth_yoy",
        "total_assets_growth_yoy",
    ],
    "lags": [f"{col}_lag1" for col in RATIO_COLS],
    "categorical": ["sector", "exchange"],
    "deltas": [
        "delta_net_margin",
        "delta_gross_margin",
        "delta_leverage",
        "delta_current_ratio",
        "delta_asset_turnover",
        "delta_ocf_to_revenue",
    ],
    # Macro không nằm trong feature groups mặc định của panel quý.
    # Chỉ thêm macro_* sau khi SAFE quarterly join đã pass leakage audit.
}


def walk_forward_splits(
    df: pd.DataFrame,
    quarter_col: str = "target_quarter_index",
    min_train_quarters: int = 4,
    include_final_test: bool = RUN_FINAL_TEST,
    final_test_target_quarter: int = FINAL_TEST_TARGET_QUARTER,
):
    '''
    Split theo TARGET quarter, không theo feature quarter.

    Ví dụ final test target 2026Q1:
      feature quarter = 2025Q4
      target quarter  = 2026Q1

    Khi include_final_test=False, target quarter 2026Q1 bị khóa hoàn toàn.
    Các target quarter sau official cutoff cũng bị loại.
    '''
    if quarter_col not in df.columns:
        raise KeyError(
            f"Thiếu quarter column cho split: {quarter_col}"
        )

    eligible = df[
        df[quarter_col].notna()
        & (
            df[quarter_col]
            <= final_test_target_quarter
        )
    ].copy()

    quarters = sorted(
        int(value)
        for value in eligible[
            quarter_col
        ].dropna().unique()
    )

    if len(quarters) <= min_train_quarters:
        print(
            f"[walk_forward_splits] Chỉ có {len(quarters)} target quarters; "
            f"cần ít nhất {min_train_quarters + 1}."
        )
        return []

    folds = []

    for i in range(min_train_quarters, len(quarters)):
        train_quarters = quarters[:i]
        validation_quarter = quarters[i]
        is_final_test = (
            validation_quarter
            == final_test_target_quarter
        )

        if is_final_test and not include_final_test:
            continue

        if validation_quarter > final_test_target_quarter:
            continue

        train_idx = eligible.index[
            eligible[quarter_col].isin(
                train_quarters
            )
        ]
        validation_idx = eligible.index[
            eligible[quarter_col]
            == validation_quarter
        ]

        folds.append(
            {
                "train_idx": train_idx,
                "val_idx": validation_idx,
                "val_quarter": validation_quarter,
                "val_quarter_label": (
                    quarter_index_to_label(
                        validation_quarter
                    )
                ),
                "feature_quarter": (
                    validation_quarter - 1
                ),
                "feature_quarter_label": (
                    quarter_index_to_label(
                        validation_quarter - 1
                    )
                ),
                "is_final_test": is_final_test,
            }
        )

    if include_final_test and not any(
        fold["is_final_test"]
        for fold in folds
    ):
        raise RuntimeError(
            "RUN_FINAL_TEST=True nhưng không tạo được final-test fold "
            f"cho target {quarter_index_to_label(final_test_target_quarter)}."
        )

    return folds

from sklearn.preprocessing import OneHotEncoder, RobustScaler


def _make_one_hot_encoder():
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=False,
        )


def fit_preprocessor(
    train_df: pd.DataFrame,
    numeric_cols: list[str],
    categorical_cols: list[str],
    scale_numeric: bool,
) -> dict:
    '''
    Fit preprocessing chỉ trên train fold.
    Feature all-null hoặc không tồn tại trong train bị loại.
    '''
    usable_numeric = [
        col
        for col in numeric_cols
        if col in train_df.columns
        and train_df[col].notna().any()
    ]
    usable_categorical = [
        col
        for col in categorical_cols
        if col in train_df.columns
        and train_df[col].notna().any()
    ]

    if not usable_numeric and not usable_categorical:
        raise ValueError(
            "Không có feature usable trong train fold."
        )

    state = {
        "numeric_cols": usable_numeric,
        "categorical_cols": usable_categorical,
        "medians": None,
        "scaler": None,
        "encoder": None,
        "output_columns": None,
        "scale_numeric": scale_numeric,
    }

    parts = []

    if usable_numeric:
        numeric = train_df[usable_numeric].apply(
            pd.to_numeric,
            errors="coerce",
        )
        medians = numeric.median()
        numeric = numeric.fillna(medians)

        if numeric.isna().any().any():
            still_null = numeric.columns[
                numeric.isna().any()
            ].tolist()
            raise ValueError(
                f"Numeric features vẫn NaN sau imputation: {still_null}"
            )

        state["medians"] = medians

        if scale_numeric:
            scaler = RobustScaler()
            numeric_array = scaler.fit_transform(numeric)
            state["scaler"] = scaler
            numeric = pd.DataFrame(
                numeric_array,
                columns=usable_numeric,
                index=train_df.index,
            )

        parts.append(numeric)

    if usable_categorical:
        categorical = (
            train_df[usable_categorical]
            .fillna("UNKNOWN")
            .astype(str)
        )
        encoder = _make_one_hot_encoder()
        encoded = encoder.fit_transform(categorical)
        encoded_df = pd.DataFrame(
            encoded,
            columns=encoder.get_feature_names_out(
                usable_categorical
            ),
            index=train_df.index,
        )
        state["encoder"] = encoder
        parts.append(encoded_df)

    matrix = pd.concat(parts, axis=1)
    state["output_columns"] = matrix.columns.tolist()

    return state


def transform_matrix(
    df: pd.DataFrame,
    state: dict,
) -> pd.DataFrame:
    parts = []

    numeric_cols = state["numeric_cols"]
    categorical_cols = state["categorical_cols"]

    if numeric_cols:
        numeric = df[numeric_cols].apply(
            pd.to_numeric,
            errors="coerce",
        )
        numeric = numeric.fillna(state["medians"])

        if state["scale_numeric"]:
            numeric_array = state["scaler"].transform(numeric)
            numeric = pd.DataFrame(
                numeric_array,
                columns=numeric_cols,
                index=df.index,
            )

        parts.append(numeric)

    if categorical_cols:
        categorical = (
            df[categorical_cols]
            .fillna("UNKNOWN")
            .astype(str)
        )
        encoded = state["encoder"].transform(categorical)
        encoded_df = pd.DataFrame(
            encoded,
            columns=state["encoder"].get_feature_names_out(
                categorical_cols
            ),
            index=df.index,
        )
        parts.append(encoded_df)

    matrix = pd.concat(parts, axis=1)
    return matrix.reindex(
        columns=state["output_columns"],
        fill_value=0,
    )


SCALED_MODEL_NAMES = {
    "linear_regression",
    "ridge",
    "lasso",
    "elastic_net",
    "huber",
    "knn",
    "linear_svm",
    "rbf_svm",
    "svr_rbf",
    "mlp",
}

# SLOW_MODEL_MAX_TRAIN_N được cấu hình ở cell đầu.
SLOW_MODEL_NAMES = {
    "linear_svm",
    "rbf_svm",
    "svr_rbf",
}


def _capped_train_data(
    X,
    y,
    model_name,
    cap=SLOW_MODEL_MAX_TRAIN_N,
    seed=42,
    stratify=False,
):
    '''
    Trả thêm train_policy để không trộn capped SVM với official leaderboard.
    '''
    if model_name not in SLOW_MODEL_NAMES or len(y) <= cap:
        return X, y, "full"

    rng = np.random.RandomState(seed)

    if stratify:
        y_array = np.asarray(y)
        selected_parts = []

        for class_value in np.unique(y_array):
            class_idx = np.where(y_array == class_value)[0]
            proportional = round(
                cap * len(class_idx) / len(y_array)
            )
            n_class = min(
                len(class_idx),
                max(5, int(proportional)),
            )
            selected_parts.append(
                rng.choice(
                    class_idx,
                    size=n_class,
                    replace=False,
                )
            )

        selected_idx = np.concatenate(selected_parts)
        rng.shuffle(selected_idx)
    else:
        selected_idx = rng.choice(
            len(y),
            size=cap,
            replace=False,
        )

    print(
        f"[cap exploratory] {model_name}: "
        f"{len(y)} -> {len(selected_idx)} train rows. "
        "Kết quả model này không thuộc official full-data leaderboard."
    )

    return (
        X[selected_idx],
        np.asarray(y)[selected_idx],
        "exploratory_capped",
    )

# ------------------------------------------------------------------
# Runtime monitoring helpers
# ------------------------------------------------------------------
MODEL_BASE_SECONDS = {
    "linear_regression": 0.5,
    "ridge": 0.5,
    "lasso": 1.0,
    "elastic_net": 1.0,
    "huber": 3.0,
    "logistic": 1.0,
    "elastic_net_logistic": 2.0,
    "knn": 0.5,
    "linear_svm": 4.0,
    "rbf_svm": 15.0,
    "svr_rbf": 15.0,
    "random_forest": 5.0,
    "extra_trees": 4.0,
    "hist_gbm": 4.0,
    "xgboost": 8.0,
    "lightgbm": 4.0,
    "catboost": 10.0,
    "mlp": 12.0,
}

FIT_RUNTIME_HISTORY = defaultdict(list)


def format_duration(seconds):
    if seconds is None or not np.isfinite(seconds):
        return "unknown"
    seconds = max(0, int(round(seconds)))
    minutes, sec = divmod(seconds, 60)
    hours, minutes = divmod(minutes, 60)
    if hours:
        return f"{hours:02d}:{minutes:02d}:{sec:02d}"
    return f"{minutes:02d}:{sec:02d}"


def selected_models(kind, factories):
    configured = MODEL_PROFILES[RUNTIME_PROFILE][kind]
    if configured is None:
        return list(factories.keys())
    return [name for name in configured if name in factories]


def estimate_fit_seconds(model_name, n_train, n_features):
    history = FIT_RUNTIME_HISTORY.get(model_name, [])
    if history:
        return float(np.median(history[-5:]))

    base = MODEL_BASE_SECONDS.get(model_name, 5.0)
    row_scale = max(0.5, n_train / 500.0)
    feature_scale = max(0.7, np.sqrt(max(1, n_features) / 20.0))

    if model_name in {"linear_svm", "rbf_svm", "svr_rbf"}:
        row_scale = min(row_scale, SLOW_MODEL_MAX_TRAIN_N / 500.0) ** 2
    elif model_name in {"random_forest", "extra_trees", "xgboost", "lightgbm", "catboost"}:
        row_scale = row_scale ** 0.8
    else:
        row_scale = row_scale ** 0.5

    return float(base * row_scale * feature_scale)


class FitHeartbeat:
    def __init__(self, label, estimated_seconds):
        self.label = label
        self.estimated_seconds = estimated_seconds
        self.started_at = None
        self.stop_event = threading.Event()
        self.thread = None

    def __enter__(self):
        self.started_at = time.perf_counter()

        if SHOW_MODEL_PROGRESS and HEARTBEAT_SECONDS > 0:
            self.thread = threading.Thread(
                target=self._run,
                daemon=True,
            )
            self.thread.start()

        return self

    def _run(self):
        while not self.stop_event.wait(HEARTBEAT_SECONDS):
            elapsed = time.perf_counter() - self.started_at
            print(
                f"  [heartbeat] {self.label} vẫn đang chạy | "
                f"elapsed={format_duration(elapsed)} | "
                f"ước tính ban đầu={format_duration(self.estimated_seconds)} | "
                f"soft budget/fold={format_duration(MODEL_SOFT_BUDGET_SECONDS)}",
                flush=True,
            )

    def __exit__(self, exc_type, exc, traceback):
        self.stop_event.set()
        if self.thread is not None:
            self.thread.join(timeout=1.0)


def print_task_runtime_plan(task_name, model_names, folds, sample, numeric, categorical):
    total_jobs = len(model_names) * len(folds)
    estimates = []
    for fold in folds:
        n_train = len(sample.loc[fold["train_idx"]])
        n_features = len(numeric) + len(categorical)
        estimates.extend(
            estimate_fit_seconds(name, n_train, n_features)
            for name in model_names
        )
    rough = float(np.sum(estimates)) if estimates else 0.0
    print(
        f"[runtime plan] task={task_name} | profile={RUNTIME_PROFILE} | "
        f"models={len(model_names)} | folds={len(folds)} | jobs={total_jobs} | "
        f"rough ETA={format_duration(rough)}",
        flush=True,
    )
    print("  models:", ", ".join(model_names), flush=True)
    return total_jobs


def progress_start(task_name, model_name, fold, job_index, total_jobs, n_train, n_features, task_started_at):
    estimated = estimate_fit_seconds(model_name, n_train, n_features)
    elapsed_task = time.perf_counter() - task_started_at
    print(
        f"[{job_index}/{total_jobs}] START {task_name} | model={model_name} | "
        f"val_q={fold['val_quarter']} | train_n={n_train} | features={n_features} | "
        f"fit ETA≈{format_duration(estimated)} | task elapsed={format_duration(elapsed_task)}",
        flush=True,
    )
    return estimated


def progress_done(task_name, model_name, fold, job_index, total_jobs, elapsed_fit, completed_jobs, task_started_at):
    FIT_RUNTIME_HISTORY[model_name].append(float(elapsed_fit))
    elapsed_task = time.perf_counter() - task_started_at
    avg_job = elapsed_task / max(1, completed_jobs)
    dynamic_eta = avg_job * max(0, total_jobs - completed_jobs)
    print(
        f"[{job_index}/{total_jobs}] DONE  {task_name} | model={model_name} | "
        f"val_q={fold['val_quarter']} | elapsed={format_duration(elapsed_fit)} | "
        f"task elapsed={format_duration(elapsed_task)} | dynamic ETA≈{format_duration(dynamic_eta)}",
        flush=True,
    )


## 4. Model registry

Các model optional được bỏ qua nếu package chưa cài. Registry có nhiều họ model để kiểm tra giả thuyết khác nhau:

- linear regularized;
- robust regression;
- nearest-neighbour/kernel;
- bagging;
- boosting;
- neural network.

XGBoost classification sẽ nhận `scale_pos_weight` theo từng train fold. SVM bị cap chỉ phục vụ runtime screening và được đánh dấu exploratory.

### Runtime profile

- `fast`: smoke test nhanh.
- `balanced`: mặc định, chạy các model chính và bỏ SVM/MLP chậm.
- `full`: chạy toàn bộ, chỉ dùng khi đã xác nhận pipeline ổn.

Mỗi fit sẽ in `START`, heartbeat mỗi 20 giây, `DONE`, thời gian thực tế và ETA động. Soft budget không thể cưỡng chế dừng một native C call đang treo, nhưng sẽ bỏ các fold còn lại của model sau khi fit hiện tại trả về. SVM đã đổi sang `probability=False`/`LinearSVC` và có `max_iter` để tránh vòng hiệu chuẩn nội bộ rất lâu.


In [26]:
from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    Lasso,
    ElasticNet,
    HuberRegressor,
    LogisticRegression,
)
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    HistGradientBoostingRegressor,
    RandomForestClassifier,
    ExtraTreesClassifier,
    HistGradientBoostingClassifier,
    IsolationForest,
)
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.svm import SVR, SVC, LinearSVC, OneClassSVM
from sklearn.neural_network import MLPRegressor, MLPClassifier

try:
    import xgboost as _xgb_check
    HAS_XGBOOST = True
except Exception as exc:
    print(
        f"[optional] xgboost chưa dùng được "
        f"({type(exc).__name__}: {exc}); bỏ qua XGBoost và ranking."
    )
    HAS_XGBOOST = False


def _xgb_reg():
    from xgboost import XGBRegressor

    return XGBRegressor(
        n_estimators=TREE_ESTIMATORS,
        max_depth=3,
        learning_rate=0.03,
        min_child_weight=8,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        objective="reg:squarederror",
        n_jobs=-1,
        random_state=RANDOM_STATE,
        verbosity=0,
    )


def _xgb_clf():
    from xgboost import XGBClassifier

    return XGBClassifier(
        n_estimators=TREE_ESTIMATORS,
        max_depth=3,
        learning_rate=0.03,
        min_child_weight=8,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        objective="binary:logistic",
        eval_metric="logloss",
        n_jobs=-1,
        random_state=RANDOM_STATE,
        verbosity=0,
    )


def _lgbm_reg():
    from lightgbm import LGBMRegressor

    return LGBMRegressor(
        random_state=RANDOM_STATE,
        verbosity=-1,
        n_estimators=TREE_ESTIMATORS,
        learning_rate=0.03,
        max_depth=4,
        num_leaves=15,
        min_child_samples=20,
        reg_alpha=0.1,
        reg_lambda=1.0,
    )


def _cat_reg():
    from catboost import CatBoostRegressor

    return CatBoostRegressor(
        random_seed=RANDOM_STATE,
        verbose=False,
        iterations=BOOST_ITERATIONS,
        learning_rate=0.03,
        depth=5,
        l2_leaf_reg=5.0,
        loss_function="RMSE",
    )


def _lgbm_clf():
    from lightgbm import LGBMClassifier

    return LGBMClassifier(
        random_state=RANDOM_STATE,
        verbosity=-1,
        n_estimators=TREE_ESTIMATORS,
        learning_rate=0.03,
        max_depth=4,
        num_leaves=15,
        min_child_samples=20,
        class_weight="balanced",
    )


def _cat_clf():
    from catboost import CatBoostClassifier

    return CatBoostClassifier(
        random_seed=RANDOM_STATE,
        verbose=False,
        iterations=BOOST_ITERATIONS,
        learning_rate=0.03,
        depth=5,
        l2_leaf_reg=5.0,
        loss_function="Logloss",
        auto_class_weights="Balanced",
    )


REGRESSION_MODEL_FACTORIES = {
    "linear_regression": lambda: LinearRegression(),
    "ridge": lambda: Ridge(alpha=1.0),
    "lasso": lambda: Lasso(
        alpha=0.01,
        max_iter=20000,
        random_state=RANDOM_STATE,
    ),
    "elastic_net": lambda: ElasticNet(
        alpha=0.01,
        l1_ratio=0.5,
        max_iter=20000,
        random_state=RANDOM_STATE,
    ),
    "huber": lambda: HuberRegressor(max_iter=2000),
    "knn": lambda: KNeighborsRegressor(
        n_neighbors=15,
        weights="distance",
    ),
    "random_forest": lambda: RandomForestRegressor(
        n_estimators=TREE_ESTIMATORS,
        max_depth=8,
        min_samples_leaf=8,
        max_features=0.7,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    ),
    "extra_trees": lambda: ExtraTreesRegressor(
        n_estimators=TREE_ESTIMATORS,
        max_depth=8,
        min_samples_leaf=8,
        max_features=0.7,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    ),
    "hist_gbm": lambda: HistGradientBoostingRegressor(
        max_depth=4,
        max_iter=BOOST_ITERATIONS,
        learning_rate=0.05,
        min_samples_leaf=15,
        l2_regularization=1.0,
        random_state=RANDOM_STATE,
    ),
    "xgboost": _xgb_reg,
    "lightgbm": _lgbm_reg,
    "catboost": _cat_reg,
    "svr_rbf": lambda: SVR(
        kernel="rbf",
        C=10.0,
        epsilon=0.01,
    ),
    "mlp": lambda: MLPRegressor(
        hidden_layer_sizes=(32, 16),
        alpha=0.01,
        max_iter=MLP_MAX_ITER,
        early_stopping=True,
        random_state=RANDOM_STATE,
    ),
}

CLASSIFICATION_MODEL_FACTORIES = {
    "logistic": lambda: LogisticRegression(
        max_iter=5000,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ),
    "elastic_net_logistic": lambda: LogisticRegression(
        penalty="elasticnet",
        solver="saga",
        l1_ratio=0.5,
        max_iter=5000,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ),
    "knn": lambda: KNeighborsClassifier(
        n_neighbors=15,
        weights="distance",
    ),
    "linear_svm": lambda: LinearSVC(
        class_weight="balanced",
        dual="auto",
        max_iter=SVM_MAX_ITER,
        random_state=RANDOM_STATE,
    ),
    "rbf_svm": lambda: SVC(
        kernel="rbf",
        probability=False,
        class_weight="balanced",
        cache_size=1024,
        max_iter=SVM_MAX_ITER,
        random_state=RANDOM_STATE,
    ),
    "random_forest": lambda: RandomForestClassifier(
        n_estimators=TREE_ESTIMATORS,
        max_depth=8,
        min_samples_leaf=8,
        max_features=0.7,
        class_weight="balanced",
        n_jobs=-1,
        random_state=RANDOM_STATE,
    ),
    "extra_trees": lambda: ExtraTreesClassifier(
        n_estimators=TREE_ESTIMATORS,
        max_depth=8,
        min_samples_leaf=8,
        max_features=0.7,
        class_weight="balanced",
        n_jobs=-1,
        random_state=RANDOM_STATE,
    ),
    "hist_gbm": lambda: HistGradientBoostingClassifier(
        max_depth=4,
        max_iter=BOOST_ITERATIONS,
        learning_rate=0.05,
        min_samples_leaf=15,
        l2_regularization=1.0,
        random_state=RANDOM_STATE,
    ),
    "xgboost": _xgb_clf,
    "lightgbm": _lgbm_clf,
    "catboost": _cat_clf,
    "mlp": lambda: MLPClassifier(
        hidden_layer_sizes=(32, 16),
        alpha=0.01,
        max_iter=MLP_MAX_ITER,
        early_stopping=True,
        random_state=RANDOM_STATE,
    ),
}

print(
    f"{len(REGRESSION_MODEL_FACTORIES)} regression models, "
    f"{len(CLASSIFICATION_MODEL_FACTORIES)} classification models; "
    f"xgboost={'có' if HAS_XGBOOST else 'không'}."
)

14 regression models, 12 classification models; xgboost=có.


## 5. Nhóm A — Supervised regression

Bốn task:

1. `net_margin_next_q`
2. `revenue_growth_qoq_next_q`
3. `ocf_signed_log_next_q`
4. `leverage_next_q`

Mỗi task có persistence baseline. Directional accuracy được tính theo:

\[
\operatorname{sign}(Y_{t+1}-Y_t)
=
\operatorname{sign}(\hat{Y}_{t+1}-Y_t)
\]

chứ không phải chỉ so dấu âm/dương của target.

In [27]:
from scipy.stats import spearmanr


def _mae(y, prediction):
    return float(np.mean(np.abs(y - prediction)))


def _rmse(y, prediction):
    return float(np.sqrt(np.mean((y - prediction) ** 2)))


def _r2(y, prediction):
    residual = np.sum((y - prediction) ** 2)
    total = np.sum((y - np.mean(y)) ** 2)
    return float(1 - residual / total) if total > 0 else float("nan")


def directional_accuracy(
    y_true,
    y_pred,
    current_values,
):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    current = np.asarray(current_values, dtype=float)

    valid = (
        np.isfinite(y_true)
        & np.isfinite(y_pred)
        & np.isfinite(current)
    )
    if valid.sum() == 0:
        return float("nan")

    true_direction = np.sign(
        y_true[valid] - current[valid]
    )
    predicted_direction = np.sign(
        y_pred[valid] - current[valid]
    )

    return float(
        np.mean(true_direction == predicted_direction)
    )


def regression_metrics(
    y_true,
    y_pred,
    current_values=None,
):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    valid = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[valid]
    y_pred = y_pred[valid]

    if len(y_true) == 0:
        return {
            "n": 0,
            "mae": np.nan,
            "rmse": np.nan,
            "r2": np.nan,
            "spearman": np.nan,
            "directional_accuracy": np.nan,
        }

    rho = (
        spearmanr(y_true, y_pred).correlation
        if len(y_true) > 2
        else float("nan")
    )

    direction = float("nan")
    if current_values is not None:
        current = np.asarray(
            current_values,
            dtype=float,
        )[valid]
        direction = directional_accuracy(
            y_true,
            y_pred,
            current,
        )

    return {
        "n": len(y_true),
        "mae": _mae(y_true, y_pred),
        "rmse": _rmse(y_true, y_pred),
        "r2": _r2(y_true, y_pred),
        "spearman": rho,
        "directional_accuracy": direction,
    }


def run_regression_task(
    panel,
    task,
    model_names=None,
    min_train_quarters=4,
):
    target = task["target"]
    numeric = task["numeric"]
    categorical = task.get(
        "categorical",
        ["sector", "exchange"],
    )

    if target not in panel.columns:
        print(f"[skip] Target không tồn tại: {target}")
        return pd.DataFrame()

    df = panel.dropna(subset=[target]).copy()
    model_names = (
        model_names
        or selected_models(
            "regression",
            REGRESSION_MODEL_FACTORIES,
        )
    )

    if RUN_FINAL_TEST:
        selected = FINAL_MODEL_SELECTION.get(task["name"])
        if not selected:
            raise RuntimeError(
                f"RUN_FINAL_TEST=True nhưng chưa khóa FINAL_MODEL_SELECTION "
                f"cho task {task['name']}."
            )
        model_names = [selected]

    folds = walk_forward_splits(
        df,
        min_train_quarters=min_train_quarters,
        include_final_test=RUN_FINAL_TEST,
    )

    if not folds:
        return pd.DataFrame()

    total_jobs = print_task_runtime_plan(
        task["name"],
        model_names,
        folds,
        df,
        numeric,
        categorical,
    )
    task_started_at = time.perf_counter()
    completed_jobs = 0
    job_index = 0
    model_elapsed = defaultdict(float)
    skipped_models = set()

    persistence_col = (
        task.get("persistence_col")
        or target.replace("_next_q", "")
    )

    fold_rows = []

    for fold in folds:
        if time.perf_counter() - task_started_at > TASK_SOFT_BUDGET_SECONDS:
            print(
                f"[task budget] Dừng task {task['name']} sau "
                f"{format_duration(time.perf_counter() - task_started_at)}; "
                "các job chưa chạy được bỏ qua.",
                flush=True,
            )
            break

        train_df = df.loc[fold["train_idx"]]
        val_df = df.loc[fold["val_idx"]]

        if train_df.empty or val_df.empty:
            continue

        y_train = train_df[target].to_numpy(dtype=float)
        y_val = val_df[target].to_numpy(dtype=float)

        if persistence_col in val_df.columns:
            persistence_prediction = val_df[
                persistence_col
            ].to_numpy(dtype=float)
            persistence_metrics = regression_metrics(
                y_val,
                persistence_prediction,
                current_values=val_df[
                    persistence_col
                ].to_numpy(dtype=float),
            )
        else:
            persistence_prediction = np.full(
                len(val_df),
                np.nanmean(y_train),
            )
            persistence_metrics = regression_metrics(
                y_val,
                persistence_prediction,
            )

        fold_rows.append(
            {
                "task": task["name"],
                "model": "persistence",
                "val_quarter": fold["val_quarter"],
                "is_final_test": fold["is_final_test"],
                "train_n": len(train_df),
                "val_n": len(val_df),
                "train_policy": "full",
                "fit_seconds": 0.0,
                **persistence_metrics,
            }
        )

        for name in model_names:
            job_index += 1

            if name in skipped_models:
                print(
                    f"[{job_index}/{total_jobs}] SKIP {task['name']} | model={name} | "
                    "đã vượt soft budget ở fold trước.",
                    flush=True,
                )
                continue

            factory = REGRESSION_MODEL_FACTORIES.get(name)
            if factory is None:
                continue

            fit_started_at = time.perf_counter()
            estimated = progress_start(
                task["name"],
                name,
                fold,
                job_index,
                total_jobs,
                len(train_df),
                len(numeric) + len(categorical),
                task_started_at,
            )

            try:
                scale_numeric = name in SCALED_MODEL_NAMES
                preprocess_state = fit_preprocessor(
                    train_df,
                    numeric,
                    categorical,
                    scale_numeric=scale_numeric,
                )

                X_train = transform_matrix(
                    train_df,
                    preprocess_state,
                )
                X_val = transform_matrix(
                    val_df,
                    preprocess_state,
                )

                X_fit, y_fit, train_policy = _capped_train_data(
                    X_train.to_numpy(dtype=float),
                    y_train,
                    name,
                )

                model = factory()
                label = (
                    f"{task['name']} / {name} / "
                    f"val_q={fold['val_quarter']}"
                )
                with FitHeartbeat(label, estimated):
                    model.fit(X_fit, y_fit)
                    prediction = model.predict(
                        X_val.to_numpy(dtype=float)
                    )

            except Exception as exc:
                elapsed_fit = time.perf_counter() - fit_started_at
                completed_jobs += 1
                progress_done(
                    task["name"], name, fold, job_index, total_jobs,
                    elapsed_fit, completed_jobs, task_started_at,
                )
                print(
                    f"[skip] {task['name']} / {name} / "
                    f"quý {fold['val_quarter']}: "
                    f"{type(exc).__name__}: {exc}",
                    flush=True,
                )
                continue

            elapsed_fit = time.perf_counter() - fit_started_at
            completed_jobs += 1
            model_elapsed[name] += elapsed_fit
            progress_done(
                task["name"], name, fold, job_index, total_jobs,
                elapsed_fit, completed_jobs, task_started_at,
            )

            if (
                SKIP_REMAINING_FOLDS_AFTER_BUDGET
                and model_elapsed[name] > MODEL_SOFT_BUDGET_SECONDS
            ):
                skipped_models.add(name)
                print(
                    f"[model budget] {name} đã dùng tổng "
                    f"{format_duration(model_elapsed[name])}; "
                    "bỏ các fold còn lại của model này.",
                    flush=True,
                )

            current_values = (
                val_df[persistence_col].to_numpy(dtype=float)
                if persistence_col in val_df.columns
                else None
            )

            fold_rows.append(
                {
                    "task": task["name"],
                    "model": name,
                    "val_quarter": fold["val_quarter"],
                    "is_final_test": fold["is_final_test"],
                    "train_n": len(train_df),
                    "val_n": len(val_df),
                    "train_policy": train_policy,
                    "fit_seconds": elapsed_fit,
                    "n_numeric_used": len(
                        preprocess_state["numeric_cols"]
                    ),
                    "n_categorical_used": len(
                        preprocess_state["categorical_cols"]
                    ),
                    **regression_metrics(
                        y_val,
                        prediction,
                        current_values=current_values,
                    ),
                }
            )

    print(
        f"[task complete] {task['name']} | elapsed="
        f"{format_duration(time.perf_counter() - task_started_at)} | "
        f"completed jobs={completed_jobs}/{total_jobs}",
        flush=True,
    )
    return pd.DataFrame(fold_rows)


REGRESSION_TASKS = [
    {
        "name": "net_margin_next_q",
        "target": "net_margin_next_q",
        "numeric": [
            "net_margin",
            "net_margin_lag1",
            "gross_margin",
            "gross_margin_lag1",
            "ocf_to_revenue",
            "ocf_to_revenue_lag1",
            "leverage",
            "leverage_lag1",
            "current_ratio",
            "current_ratio_lag1",
            "asset_turnover",
            "asset_turnover_lag1",
            "revenue_growth_yoy",
            "net_profit_growth_yoy",
            "delta_net_margin",
            "delta_gross_margin",
            "delta_leverage",
            "delta_current_ratio",
        ],
        "categorical": ["sector", "exchange"],
    },
    {
        "name": "revenue_growth_next_q",
        "target": "revenue_growth_qoq_next_q",
        "requires_flow_mode_confirmed": True,
        "persistence_col": "revenue_growth_qoq",
        "numeric": [
            "revenue_growth_qoq",
            "revenue_growth_yoy",
            "inventory",
            "total_assets",
            "asset_turnover",
            "asset_turnover_lag1",
            "gross_margin",
            "operating_cash_flow",
        ],
        "categorical": ["sector"],
    },
    {
        "name": "ocf_next_q",
        "target": "ocf_signed_log_next_q",
        "requires_flow_mode_confirmed": True,
        "persistence_col": "ocf_signed_log",
        "numeric": [
            "ocf_signed_log",
            "operating_cash_flow_lag1",
            "revenue",
            "revenue_lag1",
            "net_profit",
            "net_profit_lag1",
            "inventory",
            "inventory_lag1",
            "current_assets",
            "current_liabilities",
            "ocf_to_revenue",
            "net_margin",
        ],
        "categorical": ["sector"],
    },
    {
        "name": "leverage_next_q",
        "target": "leverage_next_q",
        "numeric": [
            "leverage",
            "leverage_lag1",
            "total_liabilities",
            "total_liabilities_lag1",
            "total_equity",
            "total_equity_lag1",
            "operating_cash_flow",
            "net_profit",
            "total_assets_growth_yoy",
            "current_ratio",
        ],
        "categorical": ["sector"],
    },
]

regression_results = {}

if not panel.empty:
    for task in REGRESSION_TASKS:
        print(f"\n=== Regression: {task['name']} ===")

        if (
            task.get("requires_flow_mode_confirmed", False)
            and not FLOW_MODE_CONFIRMED
            and not ALLOW_UNCONFIRMED_FLOW_TASKS
        ):
            print(
                "[skip flow task] FLOW_MODE_CONFIRMED=False. "
                "Xác nhận reported-quarter hay YTD cumulative trước khi chạy task này."
            )
            regression_results[task["name"]] = pd.DataFrame()
            continue

        result = run_regression_task(panel, task)
        regression_results[task["name"]] = result

        if result.empty:
            print(
                "Không đủ validation fold hoặc target coverage."
            )
            continue

        validation = result[
            ~result["is_final_test"]
        ].copy()

        official = validation[
            validation["train_policy"] == "full"
        ]

        summary = (
            official.groupby("model")[
                [
                    "mae",
                    "rmse",
                    "r2",
                    "spearman",
                    "directional_accuracy",
                ]
            ]
            .agg(["mean", "std"])
            .sort_values(("mae", "mean"))
        )

        print(
            "Official full-data validation summary "
            "(final test bị khóa):"
        )
        if summary.empty:
            print("Không có model full-data hoàn tất.")
        else:
            print(summary.round(4).to_string())

        exploratory = validation[
            validation["train_policy"]
            != "full"
        ]
        if not exploratory.empty:
            print(
                "Exploratory capped-sample models "
                "(không so sánh chính thức):"
            )
            display(
                exploratory.groupby("model")[
                    ["mae", "rmse", "r2"]
                ].mean()
            )


=== Regression: net_margin_next_q ===
[runtime plan] task=net_margin_next_q | profile=balanced | models=12 | folds=3 | jobs=36 | rough ETA=03:04
  models: linear_regression, ridge, lasso, elastic_net, huber, knn, random_forest, extra_trees, hist_gbm, xgboost, lightgbm, catboost


[1/36] START net_margin_next_q | model=linear_regression | val_q=8102 | train_n=689 | features=20 | fit ETA≈00:01 | task elapsed=00:00
[1/36] DONE  net_margin_next_q | model=linear_regression | val_q=8102 | elapsed=00:00 | task elapsed=00:00 | dynamic ETA≈00:01
[2/36] START net_margin_next_q | model=ridge | val_q=8102 | train_n=689 | features=20 | fit ETA≈00:01 | task elapsed=00:00
[2/36] DONE  net_margin_next_q | model=ridge | val_q=8102 | elapsed=00:00 | task elapsed=00:00 | dynamic ETA≈00:01
[3/36] START net_margin_next_q | model=lasso | val_q=8102 | train_n=689 | features=20 | fit ETA≈00:01 | task elapsed=00:00
[3/36] DONE  net_margin_next_q | model=lasso | val_q=8102 | elapsed=00:00 | task elapsed=00:00 | dynamic ETA≈00:01
[4/36] START net_margin_next_q | model=elastic_net | val_q=8102 | train_n=689 | features=20 | fit ETA≈00:01 | task elapsed=00:00
[4/36] DONE  net_margin_next_q | model=elastic_net | val_q=8102 | elapsed=00:00 | task elapsed=00:00 | dynamic ETA≈00:01
[5/36] START

## 6. Nhóm B — Supervised classification

Các task:

- margin tăng/giảm;
- margin âm;
- OCF âm;
- deterioration strict 3/3;
- deterioration score 3/4.

Metrics chính:

- balanced accuracy;
- F1;
- ROC-AUC;
- PR-AUC;
- Brier score.

Threshold `0.5` chỉ là mặc định ban đầu. Không được tune threshold trên final test.

Mặc định profile `balanced` **không chạy** `linear_svm`, `rbf_svm` và `mlp`. Trong profile `full`, `linear_svm` dùng `LinearSVC`, `rbf_svm` dùng `probability=False` và `max_iter`, tránh 5-fold probability calibration nội bộ từng khiến cell mất hàng chục phút.


In [28]:
from sklearn.metrics import (
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
)


def classification_metrics(
    y_true,
    y_prob,
    threshold=0.5,
):
    y_true = np.asarray(y_true, dtype=float)
    y_prob = np.asarray(y_prob, dtype=float)

    valid = np.isfinite(y_true) & np.isfinite(y_prob)
    y_true = y_true[valid]
    y_prob = y_prob[valid]

    if len(y_true) == 0:
        return {
            "n": 0,
            "prevalence": np.nan,
            "balanced_accuracy": np.nan,
            "f1": np.nan,
            "roc_auc": np.nan,
            "pr_auc": np.nan,
            "brier": np.nan,
        }

    y_pred = (y_prob >= threshold).astype(int)

    output = {
        "n": len(y_true),
        "prevalence": float(np.mean(y_true)),
        "balanced_accuracy": float(
            balanced_accuracy_score(y_true, y_pred)
        ),
        "f1": float(
            f1_score(y_true, y_pred, zero_division=0)
        ),
        "brier": float(
            brier_score_loss(y_true, y_prob)
        ),
    }

    if len(np.unique(y_true)) > 1:
        output["roc_auc"] = float(
            roc_auc_score(y_true, y_prob)
        )
        output["pr_auc"] = float(
            average_precision_score(y_true, y_prob)
        )
    else:
        output["roc_auc"] = float("nan")
        output["pr_auc"] = float("nan")

    return output


def run_classification_task(
    panel,
    task,
    model_names=None,
    min_train_quarters=4,
):
    target = task["target"]
    numeric = task["numeric"]
    categorical = task.get(
        "categorical",
        ["sector", "exchange"],
    )

    if target not in panel.columns:
        print(f"[skip] Target không tồn tại: {target}")
        return pd.DataFrame()

    df = panel.dropna(subset=[target]).copy()
    model_names = (
        model_names
        or selected_models(
            "classification",
            CLASSIFICATION_MODEL_FACTORIES,
        )
    )

    if RUN_FINAL_TEST:
        selected = FINAL_MODEL_SELECTION.get(task["name"])
        if not selected:
            raise RuntimeError(
                f"RUN_FINAL_TEST=True nhưng chưa khóa FINAL_MODEL_SELECTION "
                f"cho task {task['name']}."
            )
        model_names = [selected]

    folds = walk_forward_splits(
        df,
        min_train_quarters=min_train_quarters,
        include_final_test=RUN_FINAL_TEST,
    )

    if not folds:
        return pd.DataFrame()

    total_jobs = print_task_runtime_plan(
        task["name"],
        model_names,
        folds,
        df,
        numeric,
        categorical,
    )
    task_started_at = time.perf_counter()
    completed_jobs = 0
    job_index = 0
    model_elapsed = defaultdict(float)
    skipped_models = set()
    fold_rows = []

    for fold in folds:
        if time.perf_counter() - task_started_at > TASK_SOFT_BUDGET_SECONDS:
            print(
                f"[task budget] Dừng task {task['name']} sau "
                f"{format_duration(time.perf_counter() - task_started_at)}; "
                "các job chưa chạy được bỏ qua.",
                flush=True,
            )
            break

        train_df = df.loc[fold["train_idx"]]
        val_df = df.loc[fold["val_idx"]]

        if (
            train_df.empty
            or val_df.empty
            or train_df[target].nunique() < 2
        ):
            continue

        y_train = train_df[target].to_numpy(dtype=float)
        y_val = val_df[target].to_numpy(dtype=float)

        base_rate = float(np.mean(y_train))
        base_probability = np.full(
            len(y_val),
            base_rate,
        )

        fold_rows.append(
            {
                "task": task["name"],
                "model": "base_rate",
                "val_quarter": fold["val_quarter"],
                "is_final_test": fold["is_final_test"],
                "train_n": len(train_df),
                "val_n": len(val_df),
                "train_policy": "full",
                "fit_seconds": 0.0,
                **classification_metrics(
                    y_val,
                    base_probability,
                ),
            }
        )

        for name in model_names:
            job_index += 1

            if name in skipped_models:
                print(
                    f"[{job_index}/{total_jobs}] SKIP {task['name']} | model={name} | "
                    "đã vượt soft budget ở fold trước.",
                    flush=True,
                )
                continue

            factory = CLASSIFICATION_MODEL_FACTORIES.get(name)
            if factory is None:
                continue

            fit_started_at = time.perf_counter()
            estimated = progress_start(
                task["name"],
                name,
                fold,
                job_index,
                total_jobs,
                len(train_df),
                len(numeric) + len(categorical),
                task_started_at,
            )

            try:
                scale_numeric = name in SCALED_MODEL_NAMES
                preprocess_state = fit_preprocessor(
                    train_df,
                    numeric,
                    categorical,
                    scale_numeric=scale_numeric,
                )

                X_train = transform_matrix(
                    train_df,
                    preprocess_state,
                )
                X_val = transform_matrix(
                    val_df,
                    preprocess_state,
                )

                X_fit, y_fit, train_policy = _capped_train_data(
                    X_train.to_numpy(dtype=float),
                    y_train,
                    name,
                    stratify=True,
                )

                model = factory()

                if name == "xgboost":
                    positives = max(1, int(np.sum(y_fit == 1)))
                    negatives = max(1, int(np.sum(y_fit == 0)))
                    model.set_params(
                        scale_pos_weight=negatives / positives
                    )

                label = (
                    f"{task['name']} / {name} / "
                    f"val_q={fold['val_quarter']}"
                )
                with FitHeartbeat(label, estimated):
                    model.fit(X_fit, y_fit)

                    if hasattr(model, "predict_proba"):
                        probability = model.predict_proba(
                            X_val.to_numpy(dtype=float)
                        )[:, 1]
                    elif hasattr(model, "decision_function"):
                        score = model.decision_function(
                            X_val.to_numpy(dtype=float)
                        )
                        score = np.clip(score, -30, 30)
                        probability = 1.0 / (1.0 + np.exp(-score))
                    else:
                        probability = model.predict(
                            X_val.to_numpy(dtype=float)
                        )

            except Exception as exc:
                elapsed_fit = time.perf_counter() - fit_started_at
                completed_jobs += 1
                progress_done(
                    task["name"], name, fold, job_index, total_jobs,
                    elapsed_fit, completed_jobs, task_started_at,
                )
                print(
                    f"[skip] {task['name']} / {name} / "
                    f"quý {fold['val_quarter']}: "
                    f"{type(exc).__name__}: {exc}",
                    flush=True,
                )
                continue

            elapsed_fit = time.perf_counter() - fit_started_at
            completed_jobs += 1
            model_elapsed[name] += elapsed_fit
            progress_done(
                task["name"], name, fold, job_index, total_jobs,
                elapsed_fit, completed_jobs, task_started_at,
            )

            if (
                SKIP_REMAINING_FOLDS_AFTER_BUDGET
                and model_elapsed[name] > MODEL_SOFT_BUDGET_SECONDS
            ):
                skipped_models.add(name)
                print(
                    f"[model budget] {name} đã dùng tổng "
                    f"{format_duration(model_elapsed[name])}; "
                    "bỏ các fold còn lại của model này.",
                    flush=True,
                )

            fold_rows.append(
                {
                    "task": task["name"],
                    "model": name,
                    "val_quarter": fold["val_quarter"],
                    "is_final_test": fold["is_final_test"],
                    "train_n": len(train_df),
                    "val_n": len(val_df),
                    "train_policy": train_policy,
                    "fit_seconds": elapsed_fit,
                    "n_numeric_used": len(
                        preprocess_state["numeric_cols"]
                    ),
                    "n_categorical_used": len(
                        preprocess_state["categorical_cols"]
                    ),
                    **classification_metrics(
                        y_val,
                        probability,
                    ),
                }
            )

    print(
        f"[task complete] {task['name']} | elapsed="
        f"{format_duration(time.perf_counter() - task_started_at)} | "
        f"completed jobs={completed_jobs}/{total_jobs}",
        flush=True,
    )
    return pd.DataFrame(fold_rows)


CLASSIFICATION_TASKS = [
    {
        "name": "margin_up_next_q",
        "target": "margin_up_next_q",
        "numeric": [
            "net_margin",
            "net_margin_lag1",
            "delta_net_margin",
            "gross_margin",
            "leverage",
            "current_ratio",
            "ocf_to_revenue",
            "revenue_growth_yoy",
            "net_profit_growth_yoy",
        ],
        "categorical": ["sector"],
    },
    {
        "name": "margin_negative_next_q",
        "target": "margin_negative_next_q",
        "numeric": [
            "net_margin",
            "net_margin_lag1",
            "delta_net_margin",
            "gross_margin",
            "leverage",
            "current_ratio",
            "ocf_to_revenue",
            "revenue_growth_yoy",
            "net_profit_growth_yoy",
        ],
        "categorical": ["sector"],
    },
    {
        "name": "ocf_negative_next_q",
        "target": "ocf_negative_next_q",
        "requires_flow_mode_confirmed": True,
        "numeric": [
            "ocf_to_revenue",
            "operating_cash_flow_lag1",
            "net_profit",
            "net_profit_lag1",
            "inventory",
            "inventory_lag1",
        ],
        "categorical": ["sector"],
    },
    {
        "name": "deterioration_strict_3of3",
        "target": "deterioration_strict_3of3",
        "numeric": [
            "net_margin",
            "current_ratio",
            "leverage",
            "ocf_to_revenue",
            "delta_net_margin",
            "delta_leverage",
            "delta_current_ratio",
        ],
        "categorical": ["sector"],
    },
    {
        "name": "deterioration_3of4",
        "target": "deterioration_3of4",
        "requires_flow_mode_confirmed": True,
        "numeric": [
            "net_margin",
            "current_ratio",
            "leverage",
            "ocf_to_revenue",
            "delta_net_margin",
            "delta_leverage",
            "delta_current_ratio",
        ],
        "categorical": ["sector"],
    },
]

classification_results = {}

if not panel.empty:
    for task in CLASSIFICATION_TASKS:
        print(f"\n=== Classification: {task['name']} ===")

        if (
            task.get("requires_flow_mode_confirmed", False)
            and not FLOW_MODE_CONFIRMED
            and not ALLOW_UNCONFIRMED_FLOW_TASKS
        ):
            print(
                "[skip flow task] FLOW_MODE_CONFIRMED=False. "
                "Xác nhận reported-quarter hay YTD cumulative trước khi chạy task này."
            )
            classification_results[task["name"]] = pd.DataFrame()
            continue

        result = run_classification_task(panel, task)
        classification_results[task["name"]] = result

        if result.empty:
            print(
                "Không đủ dữ liệu, fold hoặc hai lớp để chạy."
            )
            continue

        validation = result[
            ~result["is_final_test"]
        ].copy()

        official = validation[
            validation["train_policy"] == "full"
        ]

        summary = (
            official.groupby("model")[
                [
                    "balanced_accuracy",
                    "f1",
                    "roc_auc",
                    "pr_auc",
                    "brier",
                    "prevalence",
                ]
            ]
            .agg(["mean", "std"])
            .sort_values(
                ("pr_auc", "mean"),
                ascending=False,
            )
        )

        print(
            "Official full-data validation summary "
            "(final test bị khóa):"
        )
        if summary.empty:
            print("Không có model full-data hoàn tất.")
        else:
            print(summary.round(4).to_string())

        exploratory = validation[
            validation["train_policy"]
            != "full"
        ]
        if not exploratory.empty:
            print(
                "Exploratory capped-sample models "
                "(không so sánh chính thức):"
            )
            display(
                exploratory.groupby("model")[
                    [
                        "balanced_accuracy",
                        "f1",
                        "roc_auc",
                        "pr_auc",
                        "brier",
                    ]
                ].mean()
            )


=== Classification: margin_up_next_q ===
[runtime plan] task=margin_up_next_q | profile=balanced | models=9 | folds=3 | jobs=27 | rough ETA=00:14
  models: logistic, elastic_net_logistic, knn, random_forest, extra_trees, hist_gbm, xgboost, lightgbm, catboost
[1/27] START margin_up_next_q | model=logistic | val_q=8102 | train_n=689 | features=10 | fit ETA≈00:01 | task elapsed=00:00
[1/27] DONE  margin_up_next_q | model=logistic | val_q=8102 | elapsed=00:00 | task elapsed=00:00 | dynamic ETA≈00:00
[2/27] START margin_up_next_q | model=elastic_net_logistic | val_q=8102 | train_n=689 | features=10 | fit ETA≈00:02 | task elapsed=00:00
[2/27] DONE  margin_up_next_q | model=elastic_net_logistic | val_q=8102 | elapsed=00:00 | task elapsed=00:00 | dynamic ETA≈00:02
[3/27] START margin_up_next_q | model=knn | val_q=8102 | train_n=689 | features=10 | fit ETA≈00:00 | task elapsed=00:00
[3/27] DONE  margin_up_next_q | model=knn | val_q=8102 | elapsed=00:00 | task elapsed=00:00 | dynamic ETA≈00:02


## 6.1. Calibration cho margin_up_next_q & margin_negative_next_q (Checklist 1.1)

Champion model của cả hai task là **random_forest** (pr_auc cao nhất trong
`artifacts/ml_task_suite/classification_fold_results.csv`: 0.6915 cho
`margin_up_next_q`, 0.5315 cho `margin_negative_next_q`).

Cell dưới đây lấy **out-of-fold (OOF) predicted probability** trên toàn bộ fold
validation walk-forward (`include_final_test=False` — final test **không** được
đụng ở bước này), rồi fit hai calibrator trên đúng OOF đó:

- **Platt scaling**: `LogisticRegression` trên `logit(raw_prob)`.
- **Isotonic regression**: `IsotonicRegression` trực tiếp trên `raw_prob`.

So Brier score raw / Platt / isotonic; chọn isotonic chỉ khi nó thắng rõ (margin
≥ 0.002), vì `margin_negative_next_q` chỉ ~9% positive nên isotonic dễ overfit
với mẫu ít — mặc định an toàn là Platt (đúng lưu ý kỹ thuật trong
`docs/CHECKLIST_PHASES.md` mục 1.1). Calibrator được lưu bằng `joblib` vào
`artifacts/ml_task_suite/`, tên file `calibrator_{task}_{model}.joblib`.


In [29]:
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import brier_score_loss

CALIBRATION_CHAMPION_MODEL = "random_forest"
CALIBRATION_TASK_NAMES = ["margin_up_next_q", "margin_negative_next_q"]
CALIBRATION_BRIER_IMPROVEMENT_MARGIN = 0.002


def collect_oof_probabilities(
    panel,
    task_def,
    model_name=CALIBRATION_CHAMPION_MODEL,
    min_train_quarters=4,
):
    """
    Mirror của run_classification_task, nhưng chỉ chạy 1 model và GIỮ LẠI
    raw (y_true, y_prob) từng dòng thay vì chỉ tính metric tổng hợp.
    Final-test fold luôn bị loại (include_final_test=False) — calibration
    KHÔNG được dùng final test, đúng nguyên tắc promotion gate.
    """
    target = task_def["target"]
    numeric = task_def["numeric"]
    categorical = task_def.get("categorical", ["sector", "exchange"])

    df = panel.dropna(subset=[target]).copy()
    folds = walk_forward_splits(
        df, min_train_quarters=min_train_quarters, include_final_test=False
    )

    factory = CLASSIFICATION_MODEL_FACTORIES.get(model_name)
    if factory is None:
        raise KeyError(
            f"Model '{model_name}' không có trong CLASSIFICATION_MODEL_FACTORIES."
        )

    records = []
    for fold in folds:
        train_df = df.loc[fold["train_idx"]]
        val_df = df.loc[fold["val_idx"]]
        if train_df.empty or val_df.empty or train_df[target].nunique() < 2:
            continue

        y_train = train_df[target].to_numpy(dtype=float)
        y_val = val_df[target].to_numpy(dtype=float)

        scale_numeric = model_name in SCALED_MODEL_NAMES
        preprocess_state = fit_preprocessor(
            train_df, numeric, categorical, scale_numeric=scale_numeric
        )
        X_train = transform_matrix(train_df, preprocess_state)
        X_val = transform_matrix(val_df, preprocess_state)

        X_fit, y_fit, _ = _capped_train_data(
            X_train.to_numpy(dtype=float), y_train, model_name, stratify=True
        )

        model = factory()
        model.fit(X_fit, y_fit)
        prob = model.predict_proba(X_val.to_numpy(dtype=float))[:, 1]

        tickers = (
            val_df["ticker"].to_numpy()
            if "ticker" in val_df.columns
            else np.full(len(val_df), None)
        )
        sectors = (
            val_df["sector"].to_numpy()
            if "sector" in val_df.columns
            else np.full(len(val_df), None)
        )
        # net_margin ở feature quarter (t) — dấu hiện tại, dùng cho subgroup 1.3.
        net_margin_current = (
            val_df["net_margin"].to_numpy()
            if "net_margin" in val_df.columns
            else np.full(len(val_df), np.nan)
        )
        for ticker, sector, nm, y_t, p in zip(
            tickers, sectors, net_margin_current, y_val, prob
        ):
            records.append(
                {
                    "val_quarter": fold["val_quarter"],
                    "ticker": ticker,
                    "sector": sector,
                    "net_margin_current": nm,
                    "y_true": y_t,
                    "y_prob_raw": p,
                }
            )

    return pd.DataFrame(records)


calibration_summary_rows = []

for task_def in CLASSIFICATION_TASKS:
    if task_def["name"] not in CALIBRATION_TASK_NAMES:
        continue

    print(f"\n=== Calibration OOF collection: {task_def['name']} ===")
    oof = collect_oof_probabilities(panel, task_def)

    if oof.empty:
        print("[skip] Không có OOF data (thiếu fold hoặc thiếu 2 lớp).")
        continue

    y_true = oof["y_true"].to_numpy()
    y_prob_raw = oof["y_prob_raw"].to_numpy()

    brier_raw = brier_score_loss(y_true, y_prob_raw)

    # Platt scaling: logistic regression trên logit(raw prob) — chuẩn Platt 1999.
    eps = 1e-6
    logit_raw = np.log(
        np.clip(y_prob_raw, eps, 1 - eps) / np.clip(1 - y_prob_raw, eps, 1 - eps)
    ).reshape(-1, 1)
    platt = LogisticRegression()
    platt.fit(logit_raw, y_true)
    y_prob_platt = platt.predict_proba(logit_raw)[:, 1]
    brier_platt = brier_score_loss(y_true, y_prob_platt)

    # Isotonic regression trực tiếp trên raw prob.
    isotonic = IsotonicRegression(out_of_bounds="clip")
    isotonic.fit(y_prob_raw, y_true)
    y_prob_isotonic = isotonic.predict(y_prob_raw)
    brier_isotonic = brier_score_loss(y_true, y_prob_isotonic)

    n_positive = int(y_true.sum())

    # Checklist 1.1: "nếu Brier isotonic không thắng rõ, chọn Platt cho an toàn."
    if brier_isotonic < brier_platt - CALIBRATION_BRIER_IMPROVEMENT_MARGIN:
        chosen_method = "isotonic"
        chosen_brier = brier_isotonic
        calibrator = isotonic
    else:
        chosen_method = "platt"
        chosen_brier = brier_platt
        calibrator = platt

    print(f"n={len(y_true)} | positives={n_positive} ({n_positive / len(y_true):.1%})")
    print(
        f"Brier raw={brier_raw:.4f} | Platt={brier_platt:.4f} | "
        f"Isotonic={brier_isotonic:.4f}"
    )
    print(f"-> Chọn: {chosen_method} (brier={chosen_brier:.4f})")

    calibrator_path = (
        ARTIFACT_DIR
        / f"calibrator_{task_def['name']}_{CALIBRATION_CHAMPION_MODEL}.joblib"
    )
    joblib.dump({"method": chosen_method, "model": calibrator}, calibrator_path)
    print(f"Lưu calibrator: {calibrator_path}")

    # Lưu OOF đầy đủ (raw + calibrated prob) để mục 1.2 (threshold selection)
    # đọc lại độc lập, không cần chạy lại collect_oof_probabilities().
    y_prob_calibrated = y_prob_isotonic if chosen_method == "isotonic" else y_prob_platt
    oof_out = oof.copy()
    oof_out["y_prob_calibrated"] = y_prob_calibrated
    oof_out["calibration_method"] = chosen_method
    oof_calibrated_path = ARTIFACT_DIR / f"oof_calibrated_{task_def['name']}.csv"
    oof_out.to_csv(oof_calibrated_path, index=False)
    print(f"Lưu OOF calibrated: {oof_calibrated_path}")

    calibration_summary_rows.append(
        {
            "task": task_def["name"],
            "champion_model": CALIBRATION_CHAMPION_MODEL,
            "n_oof": len(y_true),
            "n_positive": n_positive,
            "brier_raw": brier_raw,
            "brier_platt": brier_platt,
            "brier_isotonic": brier_isotonic,
            "calibration_method": chosen_method,
            "calibrator_path": str(calibrator_path),
        }
    )

calibration_summary = pd.DataFrame(calibration_summary_rows)
calibration_summary_path = ARTIFACT_DIR / "calibration_summary.csv"
calibration_summary.to_csv(calibration_summary_path, index=False)
print(f"\nĐã lưu: {calibration_summary_path}")
display(calibration_summary)



=== Calibration OOF collection: margin_up_next_q ===
n=512 | positives=273 (53.3%)
Brier raw=0.2369 | Platt=0.2313 | Isotonic=0.2247
-> Chọn: isotonic (brier=0.2247)
Lưu calibrator: /Users/mac/Desktop/Project/Thesis_Project/artifacts/ml_task_suite/calibrator_margin_up_next_q_random_forest.joblib

=== Calibration OOF collection: margin_negative_next_q ===
n=512 | positives=47 (9.2%)
Brier raw=0.0957 | Platt=0.0655 | Isotonic=0.0600
-> Chọn: isotonic (brier=0.0600)
Lưu calibrator: /Users/mac/Desktop/Project/Thesis_Project/artifacts/ml_task_suite/calibrator_margin_negative_next_q_random_forest.joblib

Đã lưu: /Users/mac/Desktop/Project/Thesis_Project/artifacts/ml_task_suite/calibration_summary.csv


,task,champion_model,n_oof,n_positive,brier_raw,brier_platt,brier_isotonic,calibration_method,calibrator_path
0,margin_up_next_q,random_forest,512,273,0.236874,0.231318,0.224696,isotonic,/Users/mac/Desktop/Project/Thesis_Project/artifacts/ml_task_suite/calibrator_margin_up_next_q_ra...
1,margin_negative_next_q,random_forest,512,47,0.095691,0.065489,0.060001,isotonic,/Users/mac/Desktop/Project/Thesis_Project/artifacts/ml_task_suite/calibrator_margin_negative_nex...


## 6.2. Chọn threshold trên validation (Checklist 1.2)

Threshold mặc định 0.50 (`CLASSIFICATION_THRESHOLDS` ở cell config) không tối ưu
cho 2 task classification. Cell dưới đây sweep threshold 0.05→0.95 (bước 0.01) trên
xác suất **đã calibrate** (`y_prob_calibrated`, lưu bởi cell 6.1), chọn theo mục
tiêu sử dụng khác nhau cho từng task, đúng mục 1.2:

- **`margin_negative_next_q`** (cảnh báo rủi ro margin âm quý sau): ưu tiên
  **recall** — bỏ sót cảnh báo tốn kém hơn cảnh báo giả. Trong các threshold đạt
  `recall ≥ 0.70`, chọn threshold có **precision cao nhất**. Nếu không threshold
  nào đạt recall 0.70 (có thể xảy ra vì chỉ 47/512 mẫu dương), chọn threshold có
  recall cao nhất đạt được và ghi rõ giới hạn này.
- **`margin_up_next_q`** (lớp cân bằng hơn, 53% dương): chọn threshold **max
  balanced accuracy** (KHÔNG dùng max F1 — xem cảnh báo dưới).

**Sự cố đã phát hiện (lần 2) — F1 chọn nhầm điểm vận hành thoái hóa:**
Ở lần chạy đầu (đã sửa lỗi mép-thấp-của-dải-đồng-hạng), threshold=0.20 được
chọn cho `margin_up_next_q` vì nó nằm giữa dải đồng hạng F1 rộng 0.05–0.35 —
NHƯNG toàn bộ dải đó là vùng thoái hóa: mọi threshold trong dải cho
`recall=1.0000, precision=0.5342` (= tỷ lệ nền), tức mô hình dự đoán
**gần như tất cả các dòng là positive**, không phân biệt được gì
(balanced_accuracy = 0.50, ngang mức đoán ngẫu nhiên — xác nhận bằng
subgroup analysis 6.3: mọi sector/margin_sign đều ≈0.50). F1 không phạt
chiến lược "luôn đoán positive" đủ mạnh khi lớp gần cân bằng, nên vẫn chọn
được điểm này dù nó vô nghĩa. Việc đổi sang chọn threshold theo max
**balanced_accuracy** (đồng nhất với metric dùng ở subgroup analysis 6.3) né
được thoái hóa này: threshold≈0.52 cho balanced_accuracy=0.6115 — thấp hơn
mức lý tưởng nhưng phản ánh đúng khả năng phân biệt thật của mô hình, không
phải một artefact của phép chọn.

Kết quả ghi vào `CLASSIFICATION_THRESHOLDS` (dùng ngay cho phần export/serving
phía sau notebook) và lưu `threshold_selection_summary.csv` +
`threshold_sweep_{task}.csv` vào ARTIFACT_DIR.


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, balanced_accuracy_score

THRESHOLD_GRID = np.round(np.arange(0.05, 0.96, 0.01), 2)
THRESHOLD_SELECTION_TARGET_RECALL = 0.70

threshold_selection_rows = []

for task_name in CALIBRATION_TASK_NAMES:
    oof_path = ARTIFACT_DIR / f"oof_calibrated_{task_name}.csv"
    if not oof_path.exists():
        print(f"[skip] Không thấy {oof_path} — chạy cell 6.1 (calibration) trước.")
        continue

    oof_cal = pd.read_csv(oof_path)
    y_true = oof_cal["y_true"].to_numpy()
    y_prob = oof_cal["y_prob_calibrated"].to_numpy()

    sweep_rows = []
    for t in THRESHOLD_GRID:
        y_pred = (y_prob >= t).astype(int)
        sweep_rows.append(
            {
                "threshold": t,
                "precision": precision_score(y_true, y_pred, zero_division=0),
                "recall": recall_score(y_true, y_pred, zero_division=0),
                "f1": f1_score(y_true, y_pred, zero_division=0),
                "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
            }
        )
    sweep = pd.DataFrame(sweep_rows)
    sweep_path = ARTIFACT_DIR / f"threshold_sweep_{task_name}.csv"
    sweep.to_csv(sweep_path, index=False)

    def pick_mid_of_tied_optimum(frame, metric_col):
        """Isotonic-calibrated probability là hàm bậc thang (ít giá trị rời
        rạc), nên nhiều threshold liên tiếp cho CÙNG MỘT tập dự đoán -> cùng
        metric. idxmax() mặc định trả về threshold ĐẦU TIÊN trong dải đồng
        hạng đó, thường rơi ngay mép thấp (gần "dự đoán tất cả positive"),
        không phải điểm vận hành có ý nghĩa. Chọn threshold Ở GIỮA dải đồng
        hạng thay vì mép thấp.
        """
        best_value = frame[metric_col].max()
        tied = frame[np.isclose(frame[metric_col], best_value)].sort_values(
            "threshold"
        )
        mid_row = tied.iloc[len(tied) // 2]
        plateau_width = float(tied["threshold"].max() - tied["threshold"].min())
        return mid_row, plateau_width, len(tied)

    if task_name == "margin_negative_next_q":
        eligible = sweep[sweep["recall"] >= THRESHOLD_SELECTION_TARGET_RECALL]
        if eligible.empty:
            # Không threshold nào đạt recall mục tiêu — chọn recall cao nhất đạt được,
            # ghi rõ giới hạn (đưa vào Limitations 6.6).
            best_row, plateau_width, plateau_n = pick_mid_of_tied_optimum(
                sweep, "recall"
            )
            reason = (
                f"KHÔNG threshold nào đạt recall≥{THRESHOLD_SELECTION_TARGET_RECALL}"
                f" (chỉ {int(y_true.sum())} mẫu dương trong {len(y_true)}); chọn "
                f"threshold có recall cao nhất đạt được (giữa dải {plateau_n} mức "
                f"đồng hạng, rộng {plateau_width:.2f})."
            )
        else:
            best_row, plateau_width, plateau_n = pick_mid_of_tied_optimum(
                eligible, "precision"
            )
            reason = (
                f"Trong các threshold đạt recall≥{THRESHOLD_SELECTION_TARGET_RECALL}, "
                f"chọn precision cao nhất (cảnh báo rủi ro margin âm ưu tiên recall); "
                f"giữa dải {plateau_n} mức đồng hạng (rộng {plateau_width:.2f}) do "
                f"isotonic step-function, lấy threshold trung tâm thay vì mép thấp. "
                f"Đã kiểm tra: điểm chọn KHÔNG nằm trong vùng thoái hóa "
                f"'luôn đoán positive' (balanced_accuracy={best_row['balanced_accuracy']:.4f}, "
                f"rõ ràng > 0.50)."
            )
    else:
        # KHÔNG dùng max F1: F1 không phạt đủ mạnh chiến lược "luôn đoán
        # positive" khi lớp gần cân bằng (~53% dương) -> chọn nhầm threshold
        # nằm trong vùng thoái hóa (balanced_accuracy=0.50, xem cell 18 &
        # phát hiện ở 6.3 subgroup analysis). Dùng max balanced_accuracy —
        # cùng metric dùng ở 6.3 — để đảm bảo threshold chọn ra phản ánh
        # khả năng phân biệt THẬT của mô hình.
        best_row, plateau_width, plateau_n = pick_mid_of_tied_optimum(
            sweep, "balanced_accuracy"
        )
        reason = (
            "Lớp cân bằng hơn — chọn threshold max BALANCED ACCURACY (không "
            "dùng max F1: F1 chọn nhầm điểm thoái hóa 'luôn đoán positive', "
            "balanced_accuracy=0.50, xem cell 18); giữa dải "
            f"{plateau_n} mức đồng hạng (rộng {plateau_width:.2f}) do isotonic "
            "step-function, lấy threshold trung tâm thay vì mép thấp."
        )

    if plateau_width > 0.10:
        print(
            f"[lưu ý] Dải đồng hạng rộng ({plateau_width:.2f}, {plateau_n} mức) — "
            "đặc tính bậc thang của isotonic calibration, không phải bug dữ liệu; "
            "đã lấy threshold trung tâm để tránh chọn nhầm mép 'gần như luôn dự "
            "đoán positive'."
        )

    chosen_threshold = float(best_row["threshold"])
    CLASSIFICATION_THRESHOLDS[task_name] = chosen_threshold

    print(f"\n=== Threshold selection: {task_name} ===")
    print(
        f"Chọn threshold={chosen_threshold:.2f} | precision={best_row['precision']:.4f} "
        f"| recall={best_row['recall']:.4f} | f1={best_row['f1']:.4f} "
        f"| balanced_accuracy={best_row['balanced_accuracy']:.4f}"
    )
    print(f"Lý do: {reason}")
    print(f"Sweep đầy đủ lưu: {sweep_path}")

    threshold_selection_rows.append(
        {
            "task": task_name,
            "chosen_threshold": chosen_threshold,
            "precision": best_row["precision"],
            "recall": best_row["recall"],
            "f1": best_row["f1"],
            "balanced_accuracy": best_row["balanced_accuracy"],
            "reason": reason,
            "sweep_path": str(sweep_path),
        }
    )

threshold_selection_summary = pd.DataFrame(threshold_selection_rows)
threshold_selection_summary_path = ARTIFACT_DIR / "threshold_selection_summary.csv"
threshold_selection_summary.to_csv(threshold_selection_summary_path, index=False)
print(f"\nĐã lưu: {threshold_selection_summary_path}")
print("CLASSIFICATION_THRESHOLDS hiện tại:", CLASSIFICATION_THRESHOLDS)
display(threshold_selection_summary)


## 6.3. Subgroup / error analysis nhanh (Checklist 1.3)

Chia kết quả OOF calibrated theo **sector** và theo **dấu net_margin hiện tại**
(quý feature `t`, trước khi dự đoán quý `t+1`), dùng threshold đã chốt ở 6.2. Mục
tiêu: phát hiện model có tệ hẳn ở một nhóm cụ thể không (ví dụ chỉ đúng với công ty
đang có margin dương). Không cố sửa gì ở bước này — chỉ phát hiện và ghi nhận; nếu
có nhóm bất thường thì đưa vào Limitations (6.6).

Ngưỡng "tệ bất thường": balanced_accuracy nhóm thấp hơn **10 điểm phần trăm** so
với balanced_accuracy tổng thể của task đó (ngưỡng thiết kế riêng, ghi rõ để có thể
tranh luận trong thesis).


In [ ]:
from sklearn.metrics import balanced_accuracy_score

SUBGROUP_MIN_N = 15  # nhóm quá nhỏ thì không đáng tin, chỉ ghi nhận n
SUBGROUP_BALANCED_ACC_GAP_FLAG = 0.10  # 10 điểm phần trăm so với tổng thể

subgroup_rows = []

for task_name in CALIBRATION_TASK_NAMES:
    oof_path = ARTIFACT_DIR / f"oof_calibrated_{task_name}.csv"
    if not oof_path.exists():
        print(f"[skip] Không thấy {oof_path} — chạy cell 6.1 trước.")
        continue

    oof_cal = pd.read_csv(oof_path)
    threshold = CLASSIFICATION_THRESHOLDS.get(task_name, 0.50)

    y_true_all = oof_cal["y_true"].to_numpy()
    y_pred_all = (oof_cal["y_prob_calibrated"].to_numpy() >= threshold).astype(int)
    overall_bal_acc = balanced_accuracy_score(y_true_all, y_pred_all)

    print(f"\n=== Subgroup analysis: {task_name} (threshold={threshold:.2f}) ===")
    print(f"Balanced accuracy tổng thể: {overall_bal_acc:.4f}")

    # --- theo sector ---
    for sector, sub in oof_cal.groupby("sector"):
        if len(sub) < SUBGROUP_MIN_N:
            continue
        y_pred_sub = (sub["y_prob_calibrated"].to_numpy() >= threshold).astype(int)
        bal_acc = balanced_accuracy_score(sub["y_true"].to_numpy(), y_pred_sub)
        gap = overall_bal_acc - bal_acc
        subgroup_rows.append(
            {
                "task": task_name,
                "group_type": "sector",
                "group_value": sector,
                "n": len(sub),
                "balanced_accuracy": bal_acc,
                "gap_vs_overall": gap,
                "flagged": gap > SUBGROUP_BALANCED_ACC_GAP_FLAG,
            }
        )

    # --- theo dấu net_margin hiện tại ---
    margin_sign = np.where(
        oof_cal["net_margin_current"] >= 0, "margin_hiện_tại_dương", "margin_hiện_tại_âm"
    )
    oof_cal_signed = oof_cal.assign(margin_sign=margin_sign)
    for sign, sub in oof_cal_signed.groupby("margin_sign"):
        if len(sub) < SUBGROUP_MIN_N:
            continue
        y_pred_sub = (sub["y_prob_calibrated"].to_numpy() >= threshold).astype(int)
        bal_acc = balanced_accuracy_score(sub["y_true"].to_numpy(), y_pred_sub)
        gap = overall_bal_acc - bal_acc
        subgroup_rows.append(
            {
                "task": task_name,
                "group_type": "margin_sign_hien_tai",
                "group_value": sign,
                "n": len(sub),
                "balanced_accuracy": bal_acc,
                "gap_vs_overall": gap,
                "flagged": gap > SUBGROUP_BALANCED_ACC_GAP_FLAG,
            }
        )

subgroup_summary = pd.DataFrame(subgroup_rows).sort_values(
    ["task", "group_type", "gap_vs_overall"], ascending=[True, True, False]
)
subgroup_summary_path = ARTIFACT_DIR / "subgroup_error_analysis.csv"
subgroup_summary.to_csv(subgroup_summary_path, index=False)
print(f"\nĐã lưu: {subgroup_summary_path}")

flagged = subgroup_summary[subgroup_summary["flagged"]]
if flagged.empty:
    print(
        "Không có nhóm nào lệch >10 điểm balanced_accuracy so với tổng thể — "
        "không phát hiện subgroup bất thường rõ rệt."
    )
else:
    print(
        f"[CẢNH BÁO] {len(flagged)} nhóm lệch >10 điểm balanced_accuracy so với "
        "tổng thể — cân nhắc ghi vào Limitations (6.6):"
    )
    display(flagged)

display(subgroup_summary)


## 7. Nhóm C — Learning to rank

Mục tiêu: xếp hạng công ty có `net_margin_gain_next_q` cao hơn trong cùng một quý.

Sửa quan trọng:

- mỗi quý train là một query riêng (`qid`);
- train rows được sort theo quý để query group liên tục;
- label training được chuyển thành relevance grade `0..4` trong từng quý;
- raw margin gain vẫn được dùng để đánh giá Spearman và top-k hit rate;
- final test tiếp tục bị khóa bởi `RUN_FINAL_TEST`.

In [30]:
from sklearn.metrics import ndcg_score


def _quarterly_relevance_grade(
    frame: pd.DataFrame,
    target: str,
    n_grades: int = 5,
) -> np.ndarray:
    percentile = frame.groupby("quarter_index")[
        target
    ].rank(method="average", pct=True)

    grade = np.floor(
        percentile * n_grades
    ).clip(0, n_grades - 1)

    return grade.astype(int).to_numpy()


def run_ranking_task(
    panel,
    target="net_margin_gain_next_q",
    numeric=None,
    categorical=("sector", "exchange"),
    min_train_quarters=4,
):
    if not HAS_XGBOOST:
        print(
            "[skip] Ranking cần xgboost; section này bị bỏ qua."
        )
        return pd.DataFrame()

    from xgboost import XGBRanker

    numeric = numeric or [
        "net_margin",
        "net_margin_lag1",
        "gross_margin",
        "leverage",
        "current_ratio",
        "asset_turnover",
        "ocf_to_revenue",
        "revenue_growth_yoy",
        "net_profit_growth_yoy",
        "delta_net_margin",
        "delta_leverage",
    ]

    if target not in panel.columns:
        return pd.DataFrame()

    df = panel.dropna(subset=[target]).copy()

    folds = walk_forward_splits(
        df,
        min_train_quarters=min_train_quarters,
        include_final_test=RUN_FINAL_TEST,
    )

    rows = []

    for fold in folds:
        train_df = (
            df.loc[fold["train_idx"]]
            .sort_values(["quarter_index", "ticker"])
            .copy()
        )
        val_df = (
            df.loc[fold["val_idx"]]
            .sort_values(["ticker"])
            .copy()
        )

        if (
            train_df.empty
            or val_df.empty
            or val_df["ticker"].nunique() < 3
        ):
            continue

        try:
            preprocess_state = fit_preprocessor(
                train_df,
                numeric,
                list(categorical),
                scale_numeric=False,
            )

            X_train = transform_matrix(
                train_df,
                preprocess_state,
            )
            X_val = transform_matrix(
                val_df,
                preprocess_state,
            )

            y_train_relevance = _quarterly_relevance_grade(
                train_df,
                target,
                n_grades=5,
            )

            # qid phải cùng nhóm liên tục; train_df đã sort theo quarter_index.
            qid_train = pd.factorize(
                train_df["quarter_index"],
                sort=True,
            )[0]

            model = XGBRanker(
                objective="rank:ndcg",
                n_estimators=300,
                max_depth=3,
                learning_rate=0.03,
                min_child_weight=8,
                subsample=0.8,
                colsample_bytree=0.8,
                reg_alpha=0.1,
                reg_lambda=1.0,
                n_jobs=-1,
                random_state=RANDOM_STATE,
                verbosity=0,
            )

            model.fit(
                X_train.to_numpy(dtype=float),
                y_train_relevance,
                qid=qid_train,
            )
            scores = model.predict(
                X_val.to_numpy(dtype=float)
            )

        except Exception as exc:
            print(
                f"[skip] ranking / quý {fold['val_quarter']}: "
                f"{type(exc).__name__}: {exc}"
            )
            continue

        y_val_raw = val_df[target].to_numpy(dtype=float)

        rho = (
            spearmanr(y_val_raw, scores).correlation
            if len(y_val_raw) > 2
            else float("nan")
        )

        # Evaluation relevance cũng là non-negative rank grade.
        val_relevance = (
            pd.Series(y_val_raw)
            .rank(method="average", pct=True)
            .mul(5)
            .floordiv(1)
            .clip(0, 4)
            .to_numpy(dtype=float)
            .reshape(1, -1)
        )
        score_matrix = scores.reshape(1, -1)

        k5 = min(5, len(y_val_raw))
        k10 = min(10, len(y_val_raw))

        ndcg5 = (
            ndcg_score(
                val_relevance,
                score_matrix,
                k=k5,
            )
            if len(y_val_raw) >= 2
            else float("nan")
        )
        ndcg10 = (
            ndcg_score(
                val_relevance,
                score_matrix,
                k=k10,
            )
            if len(y_val_raw) >= 2
            else float("nan")
        )

        order_true = np.argsort(-y_val_raw)
        order_pred = np.argsort(-scores)
        top_k = min(5, len(y_val_raw))
        hit_rate = (
            len(
                set(order_true[:top_k].tolist())
                & set(order_pred[:top_k].tolist())
            )
            / top_k
        )

        rows.append(
            {
                "val_quarter": fold["val_quarter"],
                "is_final_test": fold["is_final_test"],
                "n_companies": len(y_val_raw),
                "n_train_queries": int(
                    train_df["quarter_index"].nunique()
                ),
                "spearman": rho,
                "ndcg@5": ndcg5,
                "ndcg@10": ndcg10,
                f"top{top_k}_hit_rate": hit_rate,
            }
        )

    return pd.DataFrame(rows)


ranking_results = pd.DataFrame()

if not panel.empty:
    ranking_results = run_ranking_task(panel)

    if not ranking_results.empty:
        print(
            "Ranking validation results "
            "(final test bị khóa nếu RUN_FINAL_TEST=False):"
        )
        display(ranking_results)
        display(
            ranking_results[
                ~ranking_results["is_final_test"]
            ].mean(numeric_only=True)
            .to_frame("mean_validation")
        )
    else:
        print("Không đủ dữ liệu để chạy ranking.")

Ranking validation results (final test bị khóa nếu RUN_FINAL_TEST=False):


,val_quarter,is_final_test,n_companies,n_train_queries,spearman,ndcg@5,ndcg@10,top5_hit_rate
0,8102,False,170,4,0.310979,0.759091,0.793840,0.0
1,8103,False,172,5,0.252531,1.000000,0.911850,0.2
2,8104,False,170,6,0.063690,0.830420,0.800685,0.4


,mean_validation
val_quarter,8103.000000
is_final_test,0.000000
n_companies,170.666667
n_train_queries,5.000000
spearman,0.209067
ndcg@5,0.863170
ndcg@10,0.835458
top5_hit_rate,0.200000


## 8. Nhóm D — Unsupervised clustering

Hai mục tiêu tách biệt:

1. **Clustering diagnostics theo từng quý**: so sánh K-Means, GMM, Agglomerative và HDBSCAN.
2. **Cluster transition nhất quán**: fit một scaler + K-Means cố định trên lịch sử, sau đó dùng cùng model để gán nhãn mọi quý.

Không tính transition trực tiếp từ các cluster fit riêng từng quý vì label `0/1/2/...` có thể hoán đổi ý nghĩa.

In [31]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
    adjusted_rand_score,
)

try:
    from sklearn.cluster import HDBSCAN

    HAS_SK_HDBSCAN = True
except Exception as exc:
    print(
        f"[optional] sklearn.cluster.HDBSCAN chưa dùng được "
        f"({type(exc).__name__}: {exc}); bỏ qua HDBSCAN."
    )
    HAS_SK_HDBSCAN = False


CLUSTER_FEATURES = [
    "gross_margin",
    "net_margin",
    "leverage",
    "current_ratio",
    "asset_turnover",
    "ocf_to_revenue",
    "revenue_growth_yoy",
    "net_profit_growth_yoy",
]

CLUSTER_K_VALUES = range(2, 9)


def _prepare_unsupervised_matrix(
    frame: pd.DataFrame,
    features: list[str],
    imputer=None,
    scaler=None,
    fit=True,
):
    if fit:
        usable = [
            col
            for col in features
            if col in frame.columns
            and frame[col].notna().any()
        ]
    else:
        usable = [
            col
            for col in features
            if col in frame.columns
        ]

    if len(usable) < 2:
        raise ValueError(
            "Cần ít nhất 2 feature usable cho clustering."
        )

    numeric = frame[usable].apply(
        pd.to_numeric,
        errors="coerce",
    )

    if fit:
        imputer = SimpleImputer(strategy="median")
        scaler = RobustScaler()
        matrix = imputer.fit_transform(numeric)
        matrix = scaler.fit_transform(matrix)
    else:
        matrix = imputer.transform(numeric)
        matrix = scaler.transform(matrix)

    return matrix, usable, imputer, scaler


def _cluster_metrics(matrix, labels):
    labels = np.asarray(labels)
    non_noise = labels != -1
    unique = np.unique(labels[non_noise])

    if len(unique) < 2 or non_noise.sum() < 3:
        return {
            "silhouette": np.nan,
            "davies_bouldin": np.nan,
            "calinski_harabasz": np.nan,
            "noise_rate": float((labels == -1).mean()),
            "min_cluster_share": np.nan,
            "max_cluster_share": np.nan,
        }

    counts = pd.Series(
        labels[non_noise]
    ).value_counts(normalize=True)

    return {
        "silhouette": float(
            silhouette_score(
                matrix[non_noise],
                labels[non_noise],
            )
        ),
        "davies_bouldin": float(
            davies_bouldin_score(
                matrix[non_noise],
                labels[non_noise],
            )
        ),
        "calinski_harabasz": float(
            calinski_harabasz_score(
                matrix[non_noise],
                labels[non_noise],
            )
        ),
        "noise_rate": float((labels == -1).mean()),
        "min_cluster_share": float(counts.min()),
        "max_cluster_share": float(counts.max()),
    }


def _kmeans_bootstrap_stability(
    matrix,
    k,
    n_boot=20,
    seed=RANDOM_STATE,
):
    if len(matrix) < k * 3:
        return np.nan

    reference = KMeans(
        n_clusters=k,
        n_init=20,
        random_state=seed,
    ).fit(matrix)

    reference_labels = reference.labels_
    rng = np.random.RandomState(seed)
    scores = []

    for iteration in range(n_boot):
        idx = rng.choice(
            len(matrix),
            size=len(matrix),
            replace=True,
        )
        bootstrap_model = KMeans(
            n_clusters=k,
            n_init=10,
            random_state=seed + iteration + 1,
        ).fit(matrix[idx])

        predicted_full = bootstrap_model.predict(matrix)
        scores.append(
            adjusted_rand_score(
                reference_labels,
                predicted_full,
            )
        )

    return float(np.mean(scores))


def clustering_diagnostics_one_quarter(
    frame,
    quarter_index,
):
    rows = []
    subset = frame[
        frame["quarter_index"] == quarter_index
    ].copy()

    if len(subset) < 10:
        return pd.DataFrame()

    try:
        matrix, usable, _, _ = _prepare_unsupervised_matrix(
            subset,
            CLUSTER_FEATURES,
            fit=True,
        )
    except Exception as exc:
        print(
            f"[cluster skip] quarter={quarter_index}: "
            f"{type(exc).__name__}: {exc}"
        )
        return pd.DataFrame()

    max_k = min(max(CLUSTER_K_VALUES), len(subset) - 1)

    for k in [value for value in CLUSTER_K_VALUES if value <= max_k]:
        algorithms = {
            "kmeans": KMeans(
                n_clusters=k,
                n_init=20,
                random_state=RANDOM_STATE,
            ),
            "gmm": GaussianMixture(
                n_components=k,
                covariance_type="full",
                n_init=10,
                random_state=RANDOM_STATE,
            ),
            "agglomerative": AgglomerativeClustering(
                n_clusters=k,
                linkage="ward",
            ),
        }

        for algorithm_name, algorithm in algorithms.items():
            try:
                labels = algorithm.fit_predict(matrix)
                metric = _cluster_metrics(matrix, labels)
                rows.append(
                    {
                        "quarter_index": quarter_index,
                        "algorithm": algorithm_name,
                        "k": k,
                        "n": len(subset),
                        "n_features": len(usable),
                        "bootstrap_ari": (
                            _kmeans_bootstrap_stability(
                                matrix,
                                k,
                            )
                            if algorithm_name == "kmeans"
                            else np.nan
                        ),
                        **metric,
                    }
                )
            except Exception as exc:
                rows.append(
                    {
                        "quarter_index": quarter_index,
                        "algorithm": algorithm_name,
                        "k": k,
                        "n": len(subset),
                        "error": f"{type(exc).__name__}: {exc}",
                    }
                )

    if HAS_SK_HDBSCAN:
        try:
            hdb = HDBSCAN(
                min_cluster_size=max(5, len(subset) // 20),
            ).fit(matrix)
            rows.append(
                {
                    "quarter_index": quarter_index,
                    "algorithm": "hdbscan",
                    "k": (
                        len(set(hdb.labels_))
                        - (1 if -1 in hdb.labels_ else 0)
                    ),
                    "n": len(subset),
                    "n_features": len(usable),
                    "bootstrap_ari": np.nan,
                    **_cluster_metrics(matrix, hdb.labels_),
                }
            )
        except Exception as exc:
            print(
                f"[cluster skip] HDBSCAN quarter={quarter_index}: "
                f"{type(exc).__name__}: {exc}"
            )

    return pd.DataFrame(rows)


cluster_diagnostic_frames = []

if not panel.empty:
    for quarter_index in sorted(
        panel["quarter_index"].dropna().unique()
    ):
        result = clustering_diagnostics_one_quarter(
            panel,
            quarter_index,
        )
        if not result.empty:
            cluster_diagnostic_frames.append(result)

cluster_diagnostics = (
    pd.concat(
        cluster_diagnostic_frames,
        ignore_index=True,
    )
    if cluster_diagnostic_frames
    else pd.DataFrame()
)

if not cluster_diagnostics.empty:
    display(
        cluster_diagnostics.sort_values(
            ["quarter_index", "algorithm", "k"]
        )
    )

    latest_quarter = CURRENT_ANALYSIS_QUARTER
    development_diagnostics = cluster_diagnostics[
        cluster_diagnostics["quarter_index"] < latest_quarter
    ]

    kmeans_by_k = (
        development_diagnostics[
            development_diagnostics["algorithm"] == "kmeans"
        ]
        .groupby("k")
        .agg(
            mean_silhouette=("silhouette", "mean"),
            mean_bootstrap_ari=("bootstrap_ari", "mean"),
            mean_db=("davies_bouldin", "mean"),
            mean_min_cluster_share=("min_cluster_share", "mean"),
            mean_max_cluster_share=("max_cluster_share", "mean"),
            quarters=("quarter_index", "nunique"),
        )
        .reset_index()
        .sort_values(
            ["mean_silhouette", "mean_bootstrap_ari"],
            ascending=False,
        )
    )

    print("K-Means development diagnostics:")
    display(kmeans_by_k)

    balanced_k_candidates = kmeans_by_k[
        (kmeans_by_k["mean_min_cluster_share"] >= 0.05)
        & (kmeans_by_k["quarters"] >= 3)
    ].copy()

    if balanced_k_candidates.empty:
        print(
            "[cluster warning] Không có K nào vừa ổn định vừa giữ "
            "mỗi cluster >=5% doanh nghiệp. Clustering chỉ exploratory."
        )
        SELECTED_CLUSTER_K = 4
    else:
        SELECTED_CLUSTER_K = int(
            balanced_k_candidates.iloc[0]["k"]
        )
else:
    SELECTED_CLUSTER_K = 4

print("SELECTED_CLUSTER_K:", SELECTED_CLUSTER_K)


def fit_fixed_cluster_model(
    frame: pd.DataFrame,
    selected_k: int,
):
    latest_quarter = CURRENT_ANALYSIS_QUARTER
    train = frame[
        frame["quarter_index"] < latest_quarter
    ].copy()

    if len(train) < selected_k * 3:
        print(
            "[cluster transition] Không đủ historical rows "
            "để fit fixed clustering model."
        )
        return None

    matrix_train, usable, imputer, scaler = (
        _prepare_unsupervised_matrix(
            train,
            CLUSTER_FEATURES,
            fit=True,
        )
    )

    model = KMeans(
        n_clusters=selected_k,
        n_init=30,
        random_state=RANDOM_STATE,
    ).fit(matrix_train)

    all_valid = frame[
        frame[usable].notna().any(axis=1)
    ].copy()

    matrix_all, _, _, _ = _prepare_unsupervised_matrix(
        all_valid,
        usable,
        imputer=imputer,
        scaler=scaler,
        fit=False,
    )

    assignments = all_valid[
        [
            "ticker",
            "quarter_index",
            "fiscal_year",
            "quarter_num",
            "sector",
        ]
    ].copy()
    assignments["cluster"] = model.predict(matrix_all)

    return {
        "model": model,
        "imputer": imputer,
        "scaler": scaler,
        "features": usable,
        "assignments": assignments,
        "train_max_quarter": int(
            train["quarter_index"].max()
        ),
    }


fixed_cluster_bundle = (
    fit_fixed_cluster_model(
        panel,
        SELECTED_CLUSTER_K,
    )
    if not panel.empty
    else None
)

fixed_cluster_assignments = (
    fixed_cluster_bundle["assignments"]
    if fixed_cluster_bundle is not None
    else pd.DataFrame()
)

if not fixed_cluster_assignments.empty:
    print(
        "Fixed-model cluster assignments; "
        f"fit through quarter {fixed_cluster_bundle['train_max_quarter']}."
    )
    display(fixed_cluster_assignments.head(20))

,quarter_index,algorithm,k,n,n_features,bootstrap_ari,silhouette,davies_bouldin,calinski_harabasz,noise_rate,min_cluster_share,max_cluster_share
2,8097,agglomerative,2,173,6,NaN,0.845385,0.098431,49.239749,0.0,0.005780,0.994220
5,8097,agglomerative,3,173,6,NaN,0.677430,0.471939,55.263882,0.0,0.005780,0.965318
8,8097,agglomerative,4,173,6,NaN,0.672532,0.579086,64.477122,0.0,0.005780,0.936416
11,8097,agglomerative,5,173,6,NaN,0.264876,0.944649,66.129027,0.0,0.005780,0.583815
14,8097,agglomerative,6,173,6,NaN,0.272704,0.741185,68.760859,0.0,0.005780,0.583815
...,...,...,...,...,...,...,...,...,...,...,...,...
182,8105,kmeans,4,170,8,0.518827,0.415578,0.955953,59.972460,0.0,0.005882,0.823529
185,8105,kmeans,5,170,8,0.504262,0.463095,0.881060,60.016497,0.0,0.005882,0.847059
188,8105,kmeans,6,170,8,0.690892,0.359583,0.948718,62.500025,0.0,0.005882,0.711765
191,8105,kmeans,7,170,8,0.699272,0.373920,0.871080,60.863102,0.0,0.005882,0.723529


K-Means development diagnostics:


,k,mean_silhouette,mean_bootstrap_ari,mean_db,mean_min_cluster_share,mean_max_cluster_share,quarters
0,2,0.890955,0.685651,0.266018,0.012251,0.987749,8
1,3,0.795558,0.661743,0.345560,0.007899,0.965515,8
2,4,0.697953,0.598612,0.389125,0.006462,0.913922,8
3,5,0.594709,0.658454,0.508062,0.006462,0.858395,8
4,6,0.539258,0.644753,0.540947,0.005752,0.812024,8
5,7,0.421620,0.632189,0.610316,0.005752,0.724245,8
6,8,0.410475,0.589899,0.605399,0.005752,0.711214,8


[cluster warning] Không có K nào vừa ổn định vừa giữ mỗi cluster >=5% doanh nghiệp. Clustering chỉ exploratory.
SELECTED_CLUSTER_K: 4
Fixed-model cluster assignments; fit through quarter 8104.


,ticker,quarter_index,fiscal_year,quarter_num,sector,cluster
0,AAA,8097,2024,1,Nguyên vật liệu,0
1,AAA,8098,2024,2,Nguyên vật liệu,0
2,AAA,8099,2024,3,Nguyên vật liệu,0
3,AAA,8100,2024,4,Nguyên vật liệu,0
4,AAA,8101,2025,1,Nguyên vật liệu,0
5,AAA,8102,2025,2,Nguyên vật liệu,0
6,AAA,8103,2025,3,Nguyên vật liệu,0
7,AAA,8104,2025,4,Nguyên vật liệu,0
8,AAA,8105,2026,1,Nguyên vật liệu,0
9,AAAA,8102,2025,2,Chưa phân loại,0


In [32]:
# Transition matrix dùng nhãn từ CÙNG một fixed K-Means model.
# Vì vậy cluster 0/1/... có cùng định nghĩa trong mọi quý.

if fixed_cluster_assignments.empty:
    print(
        "Chưa có fixed cluster assignments để tính transition."
    )
else:
    quarters_sorted = sorted(
        fixed_cluster_assignments[
            "quarter_index"
        ].unique()
    )

    transition_frames = []

    for q1, q2 in zip(
        quarters_sorted[:-1],
        quarters_sorted[1:],
    ):
        if q2 - q1 != 1:
            continue

        first = fixed_cluster_assignments[
            fixed_cluster_assignments["quarter_index"] == q1
        ][["ticker", "cluster"]].rename(
            columns={"cluster": "cluster_t"}
        )

        second = fixed_cluster_assignments[
            fixed_cluster_assignments["quarter_index"] == q2
        ][["ticker", "cluster"]].rename(
            columns={"cluster": "cluster_t1"}
        )

        merged = first.merge(
            second,
            on="ticker",
            how="inner",
        )

        if merged.empty:
            continue

        transition = pd.crosstab(
            merged["cluster_t"],
            merged["cluster_t1"],
            normalize="index",
        )

        print(
            f"\nP(cluster_{q2} | cluster_{q1}), "
            f"n={len(merged)} ticker:"
        )
        display(transition.round(3))

        long_form = (
            transition.stack()
            .rename("probability")
            .reset_index()
        )
        long_form["quarter_from"] = q1
        long_form["quarter_to"] = q2
        long_form["n_common_tickers"] = len(merged)
        transition_frames.append(long_form)

    cluster_transitions = (
        pd.concat(
            transition_frames,
            ignore_index=True,
        )
        if transition_frames
        else pd.DataFrame()
    )


P(cluster_8098 | cluster_8097), n=173 ticker:


cluster_t1,0
cluster_t,
0,1.0



P(cluster_8099 | cluster_8098), n=172 ticker:


cluster_t1,0
cluster_t,
0,1.0



P(cluster_8100 | cluster_8099), n=174 ticker:


...

cluster_t1,0,1,2
cluster_t,,,
0,0.977,0.012,0.012
2,1.000,0.000,0.000



P(cluster_8104 | cluster_8103), n=171 ticker:


cluster_t1,0,2
cluster_t,,
0,0.97,0.03
1,1.00,0.00
2,1.00,0.00



P(cluster_8105 | cluster_8104), n=166 ticker:


cluster_t1,0
cluster_t,
0,1.0
2,1.0


## 9. Nhóm E — Unsupervised anomaly detection

Thiết kế production-like:

- fit preprocessing và detector trên các quý lịch sử;
- score riêng quý mới nhất;
- dùng sector-relative robust z-score;
- `contamination=0.05` chỉ là policy để ưu tiên top candidates, không phải ground-truth rằng 5% công ty chắc chắn bất thường.

Các detector:

- Isolation Forest;
- Local Outlier Factor với `novelty=True`;
- One-Class SVM;
- Elliptic Envelope.

In [33]:
from sklearn.neighbors import LocalOutlierFactor
from sklearn.covariance import EllipticEnvelope

ANOMALY_FEATURES = [
    f"{col}_sector_z"
    for col in RATIO_COLS
] + [
    "delta_net_margin",
    "delta_leverage",
]


def fit_and_score_latest_anomalies(
    panel,
    features=ANOMALY_FEATURES,
    contamination=0.05,
):
    usable = [
        col
        for col in features
        if col in panel.columns
        and panel[col].notna().any()
    ]

    if len(usable) < 2:
        print(
            "[anomaly] Cần ít nhất 2 feature usable."
        )
        return pd.DataFrame(), None

    latest_quarter = CURRENT_ANALYSIS_QUARTER
    history = panel[
        panel["quarter_index"] < latest_quarter
    ].copy()
    current = panel[
        panel["quarter_index"] == latest_quarter
    ].copy()

    if len(history) < 30 or len(current) < 5:
        print(
            f"[anomaly] history={len(history)}, current={len(current)}; "
            "chưa đủ để score đáng tin cậy."
        )
        return pd.DataFrame(), None

    imputer = SimpleImputer(strategy="median")
    scaler = RobustScaler()

    X_history = imputer.fit_transform(
        history[usable].apply(
            pd.to_numeric,
            errors="coerce",
        )
    )
    X_history = scaler.fit_transform(X_history)

    X_current = imputer.transform(
        current[usable].apply(
            pd.to_numeric,
            errors="coerce",
        )
    )
    X_current = scaler.transform(X_current)

    output = current[
        [
            "ticker",
            "fiscal_year",
            "quarter_num",
            "quarter_index",
            "sector",
        ]
    ].copy()

    models = {}

    isolation = IsolationForest(
        n_estimators=500,
        contamination=contamination,
        random_state=RANDOM_STATE,
    ).fit(X_history)
    output["iso_score"] = -isolation.score_samples(
        X_current
    )
    models["isolation_forest"] = isolation

    n_neighbors = min(
        20,
        max(5, len(history) - 1),
    )
    lof = LocalOutlierFactor(
        n_neighbors=n_neighbors,
        contamination=contamination,
        novelty=True,
    ).fit(X_history)
    output["lof_score"] = -lof.score_samples(
        X_current
    )
    models["lof"] = lof

    try:
        one_class_svm = OneClassSVM(
            nu=contamination,
            kernel="rbf",
            gamma="scale",
        ).fit(X_history)
        output["ocsvm_score"] = -one_class_svm.decision_function(
            X_current
        )
        models["one_class_svm"] = one_class_svm
    except Exception as exc:
        print(
            f"[skip] OneClassSVM: "
            f"{type(exc).__name__}: {exc}"
        )
        output["ocsvm_score"] = np.nan

    try:
        elliptic = EllipticEnvelope(
            contamination=contamination,
            random_state=RANDOM_STATE,
        ).fit(X_history)
        output["elliptic_score"] = -elliptic.decision_function(
            X_current
        )
        models["elliptic_envelope"] = elliptic
    except Exception as exc:
        print(
            f"[skip] EllipticEnvelope: "
            f"{type(exc).__name__}: {exc}"
        )
        output["elliptic_score"] = np.nan

    # Percentile rank để kết hợp score có scale khác nhau.
    score_cols = [
        col
        for col in [
            "iso_score",
            "lof_score",
            "ocsvm_score",
            "elliptic_score",
        ]
        if output[col].notna().any()
    ]

    for col in score_cols:
        output[f"{col}_pct"] = output[col].rank(
            pct=True,
            method="average",
        )

    percentile_cols = [
        f"{col}_pct"
        for col in score_cols
    ]
    output["ensemble_anomaly_score"] = output[
        percentile_cols
    ].mean(axis=1)

    for col in usable:
        output[col] = current[col].to_numpy()

    output = output.sort_values(
        "ensemble_anomaly_score",
        ascending=False,
    ).reset_index(drop=True)

    bundle = {
        "features": usable,
        "imputer": imputer,
        "scaler": scaler,
        "models": models,
        "history_max_quarter": int(
            history["quarter_index"].max()
        ),
        "scored_quarter": int(latest_quarter),
    }

    return output, bundle


latest_anomaly_scores = pd.DataFrame()
anomaly_bundle = None

if not panel.empty:
    latest_anomaly_scores, anomaly_bundle = (
        fit_and_score_latest_anomalies(panel)
    )

    if not latest_anomaly_scores.empty:
        print(
            "Top anomaly candidates của quý mới nhất; "
            "cần audit thủ công và đối chiếu tài liệu."
        )
        display(latest_anomaly_scores.head(20))

Top anomaly candidates của quý mới nhất; cần audit thủ công và đối chiếu tài liệu.


,ticker,fiscal_year,quarter_num,quarter_index,sector,iso_score,lof_score,ocsvm_score,elliptic_score,iso_score_pct,lof_score_pct,ocsvm_score_pct,elliptic_score_pct,ensemble_anomaly_score,leverage_sector_z,current_ratio_sector_z,gross_margin_sector_z,net_margin_sector_z,asset_turnover_sector_z,ocf_to_revenue_sector_z,delta_net_margin,delta_leverage
0,BTT,2026,1,8105,Thương mại hàng không thiết yếu,0.684183,7.703056,6.134721,42454.333944,1.000000,1.000000,1.000000,1.000000,1.000000,-1.549951,7.651666,18.708527,132.764124,-0.513991,2.163331,0.930505,-0.006272
1,AGM,2026,1,8105,"Thực phẩm, đồ uống và thuốc lá",0.596474,5.544968,5.980104,10845.343199,0.994118,0.988235,0.988235,0.994118,0.991176,2.203812,-2.284184,-6.421353,-64.980707,-1.447908,-3.354193,NaN,NaN
2,AAM,2026,1,8105,"Thực phẩm, đồ uống và thuốc lá",0.549529,3.854612,6.020377,9303.699654,0.982353,0.976471,0.994118,0.988235,0.985294,-1.285899,55.142702,0.077238,0.531446,-0.578269,1.844280,NaN,NaN
3,CVN,2026,1,8105,"Thực phẩm, đồ uống và thuốc lá",0.501296,6.604879,5.584772,5389.237290,0.964706,0.994118,0.982353,0.982353,0.980882,-0.972931,-1.679940,-0.508592,-4.256264,-1.400639,0.250721,3.781681,0.000716
4,CPI,2026,1,8105,Vận tải,0.553335,2.539821,0.389588,-209.752414,0.988235,0.964706,0.947059,0.894118,0.948529,4.189312,-0.954570,-1.300802,-0.725325,5.855370,-0.848707,-0.017152,-0.098070
5,CMI,2026,1,8105,Nguyên vật liệu,0.528695,1.999714,0.330470,6.088829,0.970588,0.911765,0.941176,0.964706,0.947059,5.527659,-0.930878,-2.564922,-10.101741,-0.632042,NaN,-0.410284,0.028832
6,CPA,2026,1,8105,"Thực phẩm, đồ uống và thuốc lá",0.539281,1.627707,2.229321,-47.851744,0.976471,0.847059,0.976471,0.958824,0.939706,0.370524,-1.697244,3.811541,0.364674,-1.306868,-8.589029,0.711359,0.008036
7,CIG,2026,1,8105,Hàng hóa công nghiệp,0.410457,2.195762,1.481770,29.246781,0.823529,0.952941,0.970588,0.970588,0.929412,0.633758,0.790461,0.273050,-0.426756,-0.999404,-0.522569,-0.935838,0.021417
8,BHA,2026,1,8105,Tiện ích,0.447652,1.779864,0.637448,-124.658994,0.905882,0.870588,0.958824,0.941176,0.919118,-0.172517,-0.353143,-0.548292,-1.091293,-0.547033,2.468453,-0.686026,-0.052552
9,AGF,2026,1,8105,"Thực phẩm, đồ uống và thuốc lá",0.475419,2.146472,-0.558001,-253.137332,0.958824,0.941176,0.888235,0.870588,0.914706,3.508208,-2.042535,-1.119306,-2.557803,0.846352,0.112843,-0.138953,0.141461


In [34]:
def synthetic_corruption_test(
    panel,
    features=ANOMALY_FEATURES,
    n_inject=15,
    seed=RANDOM_STATE,
):
    '''
    Fit detector trên lịch sử, sau đó thêm các bản sao bị làm hỏng vào
    quý mới nhất. Detector tốt phải đưa nhiều mẫu synthetic vào top rank.
    '''
    usable = [
        col
        for col in features
        if col in panel.columns
        and panel[col].notna().any()
    ]

    if len(usable) < 2:
        return pd.DataFrame()

    latest_quarter = panel["quarter_index"].max()
    history = panel[
        panel["quarter_index"] < latest_quarter
    ].copy()
    current = panel[
        panel["quarter_index"] == latest_quarter
    ].copy()

    if len(history) < 30 or len(current) < 10:
        print(
            "Không đủ history/current rows để chạy synthetic test."
        )
        return pd.DataFrame()

    rng = np.random.RandomState(seed)
    n_inject = min(n_inject, len(current))

    selected = current.sample(
        n=n_inject,
        random_state=seed,
    ).copy()
    selected["_is_synthetic"] = True

    original_current = current.copy()
    original_current["_is_synthetic"] = False

    for position, idx in enumerate(selected.index):
        kind = position % 4

        if kind == 0 and "leverage_sector_z" in selected.columns:
            selected.loc[idx, "leverage_sector_z"] = (
                selected.loc[idx, "leverage_sector_z"] * 5
            )
        elif kind == 1 and "ocf_to_revenue_sector_z" in selected.columns:
            selected.loc[idx, "ocf_to_revenue_sector_z"] = (
                -selected.loc[idx, "ocf_to_revenue_sector_z"]
                - 5
            )
        elif kind == 2 and "current_ratio_sector_z" in selected.columns:
            selected.loc[idx, "current_ratio_sector_z"] = -5
        elif "delta_net_margin" in selected.columns:
            selected.loc[idx, "delta_net_margin"] = (
                selected.loc[idx, "delta_net_margin"] * 20
                - 5
            )

    evaluation_current = pd.concat(
        [original_current, selected],
        ignore_index=True,
    )

    imputer = SimpleImputer(strategy="median")
    scaler = RobustScaler()

    X_history = imputer.fit_transform(
        history[usable].apply(
            pd.to_numeric,
            errors="coerce",
        )
    )
    X_history = scaler.fit_transform(X_history)

    X_evaluation = imputer.transform(
        evaluation_current[usable].apply(
            pd.to_numeric,
            errors="coerce",
        )
    )
    X_evaluation = scaler.transform(X_evaluation)

    detector = IsolationForest(
        n_estimators=500,
        contamination=0.05,
        random_state=seed,
    ).fit(X_history)

    scored = evaluation_current[
        [
            "ticker",
            "quarter_index",
            "sector",
            "_is_synthetic",
        ]
    ].copy()
    scored["iso_score"] = -detector.score_samples(
        X_evaluation
    )
    scored = scored.sort_values(
        "iso_score",
        ascending=False,
    ).reset_index(drop=True)

    k = len(selected)
    recall_at_2k = (
        scored.head(min(2 * k, len(scored)))[
            "_is_synthetic"
        ].sum()
        / k
    )

    print(
        f"Synthetic anomalies={k}; "
        f"IsolationForest recall@{min(2 * k, len(scored))}="
        f"{recall_at_2k:.3f}"
    )

    scored.attrs["synthetic_recall_at_2k"] = float(
        recall_at_2k
    )
    return scored


synthetic_result = pd.DataFrame()

if not panel.empty:
    synthetic_result = synthetic_corruption_test(panel)

Synthetic anomalies=15; IsolationForest recall@30=0.467


## 10. Nhóm F — Causal / relational analysis

Causal branch mặc định **không chạy** (`RUN_CAUSAL=False`).

Lý do:

- chỉ có ít quý;
- hidden confounding lớn;
- treatment liên tục không phù hợp với T-learner nhị phân;
- outcome cùng kỳ dễ reverse causality.

Bản sửa dùng:

- treatment tại quý `t`;
- outcome tại quý `t+1`;
- pre-treatment covariates tại `t`;
- continuous treatment → `LinearDML`;
- T-learner chỉ dùng khi có treatment nhị phân thật như `auditor_changed`.

Kết quả vẫn là observational causal estimate và phải đi kèm assumptions/CI/sample size.

In [35]:
from src.causal.treatment_effect import (
    estimate_ate,
    estimate_ate_dml,
)


def continuous_causal_gate(
    frame,
    treatment_col,
    outcome_col,
    covariates,
    min_n=200,
):
    required = [
        treatment_col,
        outcome_col,
        *covariates,
    ]
    missing = [
        col for col in required
        if col not in frame.columns
    ]

    if missing:
        return {
            "ok": False,
            "reason": "missing_columns",
            "missing": missing,
        }

    sample = frame[required].dropna()

    if len(sample) < min_n:
        return {
            "ok": False,
            "reason": "insufficient_sample",
            "n": len(sample),
            "minimum": min_n,
        }

    treatment_std = float(
        sample[treatment_col].std()
    )
    if not np.isfinite(treatment_std) or treatment_std <= 1e-12:
        return {
            "ok": False,
            "reason": "no_treatment_variation",
            "n": len(sample),
        }

    return {
        "ok": True,
        "reason": "ok",
        "n": len(sample),
        "treatment_std": treatment_std,
    }


CONTINUOUS_CAUSAL_QUESTIONS = [
    {
        "treatment_col": "leverage",
        "outcome_col": "net_margin_next_q",
        "covariates": (
            "net_margin",
            "gross_margin",
            "current_ratio",
            "ocf_to_revenue",
            "asset_turnover",
            "quarter_index",
        ),
    },
    {
        "treatment_col": "revenue_growth_qoq",
        "outcome_col": "net_margin_next_q",
        "covariates": (
            "net_margin",
            "gross_margin",
            "leverage",
            "current_ratio",
            "ocf_to_revenue",
            "quarter_index",
        ),
    },
    {
        "treatment_col": "current_ratio",
        "outcome_col": "ocf_to_revenue_next_q",
        "covariates": (
            "ocf_to_revenue",
            "net_margin",
            "gross_margin",
            "leverage",
            "asset_turnover",
            "quarter_index",
        ),
    },
]


BINARY_CAUSAL_TEMPLATES = [
    {
        "treatment_col": "auditor_changed",
        "outcome_col": "net_margin_next_q",
        "covariates": (
            "net_margin",
            "gross_margin",
            "leverage",
            "current_ratio",
            "ocf_to_revenue",
            "quarter_index",
        ),
    }
]


if not RUN_CAUSAL:
    print(
        "Causal section đang khóa. "
        "Đặt RUN_CAUSAL=True sau khi kiểm tra sample/overlap/assumptions."
    )
elif panel.empty:
    print("Panel rỗng; không chạy causal.")
else:
    # Continuous treatment: chỉ LinearDML, không dùng T-learner.
    for question in CONTINUOUS_CAUSAL_QUESTIONS:
        gate = continuous_causal_gate(
            panel,
            **question,
        )
        print(
            "\nContinuous causal gate:",
            question["treatment_col"],
            "->",
            question["outcome_col"],
            gate,
        )

        if not gate["ok"]:
            continue

        result = estimate_ate_dml(
            panel=panel,
            **question,
        )
        print("LinearDML result:", result)

    # Binary treatment: chỉ chạy khi cột thật sự tồn tại.
    for question in BINARY_CAUSAL_TEMPLATES:
        treatment = question["treatment_col"]

        if treatment not in panel.columns:
            print(
                f"[binary causal skip] {treatment} chưa có trong panel."
            )
            continue

        required = [
            treatment,
            question["outcome_col"],
            *question["covariates"],
        ]
        sample = panel[required].dropna()

        treated = int((sample[treatment] == 1).sum())
        control = int((sample[treatment] == 0).sum())

        if treated < 30 or control < 100:
            print(
                f"[binary causal skip] {treatment}: "
                f"treated={treated}, control={control}; không đủ mẫu."
            )
            continue

        result_t = estimate_ate(
            panel=panel,
            **question,
        )
        result_dml = estimate_ate_dml(
            panel=panel,
            **question,
        )

        print("T-learner result:", result_t)
        print("DML robustness result:", result_dml)

Causal section đang khóa. Đặt RUN_CAUSAL=True sau khi kiểm tra sample/overlap/assumptions.


## 11. Dimensionality reduction (PCA + t-SNE/UMAP)

Khong phai model du bao - dung de EDA va ho tro dien giai cac cum o Section 8.
`umap-learn` KHONG nam trong `requirements.txt` (optional) - notebook tu dong
dung t-SNE (co san trong scikit-learn) thay the neu chua cai UMAP.

In [36]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE

if not panel.empty:
    pca_df = panel.dropna(subset=CLUSTER_FEATURES).copy()
    if len(pca_df) >= 10:
        X_pca = StandardScaler().fit_transform(pca_df[CLUSTER_FEATURES])
        pca = PCA(n_components=min(4, len(CLUSTER_FEATURES)), random_state=42).fit(X_pca)
        loadings = pd.DataFrame(pca.components_.T, index=CLUSTER_FEATURES,
                                 columns=[f"PC{i + 1}" for i in range(pca.n_components_)])
        print("Ty le phuong sai giai thich moi PC:", np.round(pca.explained_variance_ratio_, 3))
        print("Loadings (KHONG tu dat ten factor neu loadings khong ho tro ro rang):")
        display(loadings.round(3))

        if HAS_UMAP:
            import umap
            emb = umap.UMAP(random_state=42).fit_transform(X_pca)
            print("UMAP embedding shape:", emb.shape)
        else:
            perplexity = min(30, max(5, len(pca_df) // 10))
            emb = TSNE(n_components=2, random_state=42, perplexity=perplexity).fit_transform(X_pca)
            print(f"UMAP chua duoc cai (pip install umap-learn) - dung t-SNE thay the (perplexity={perplexity}), "
                  f"shape: {emb.shape}")
    else:
        print(f"Chi {len(pca_df)} dong du CLUSTER_FEATURES - can >= 10 de chay PCA co y nghia.")

Ty le phuong sai giai thich moi PC: [0.279 0.154 0.139 0.127]
Loadings (KHONG tu dat ten factor neu loadings khong ho tro ro rang):


,PC1,PC2,PC3,PC4
gross_margin,0.538,-0.112,-0.349,-0.129
net_margin,0.499,-0.096,-0.336,0.056
leverage,-0.439,-0.059,-0.493,-0.057
current_ratio,0.387,0.112,0.583,0.107
asset_turnover,-0.307,-0.201,0.194,0.496
ocf_to_revenue,0.103,-0.607,0.130,0.352
revenue_growth_yoy,0.018,0.706,0.049,0.158
net_profit_growth_yoy,0.117,0.234,-0.353,0.756


UMAP chua duoc cai (pip install umap-learn) - dung t-SNE thay the (perplexity=28), shape: (281, 2)


## 12. Association-rule mining (thu cong, khong can `mlxtend`)

`mlxtend` khong nam trong `requirements.txt`. Vi tap luat can xet o day nho
(vai bien roi rac hoa), tu viet ham tinh support/confidence/lift truc tiep
bang pandas thay vi them 1 dependency moi chi cho viec nay - ket qua giong het
Apriori/FP-Growth cho tap luat nho, chi khong scale duoc len hang nghin item
(khong can thiet o quy mo du lieu hien tai).

In [37]:
def discretize_for_rules(panel: pd.DataFrame) -> pd.DataFrame:
    df = panel.copy()
    df["leverage_bin"] = pd.qcut(df["leverage"], q=[0, .33, .67, 1.0], labels=["LOW", "MED", "HIGH"], duplicates="drop")
    df["current_ratio_bin"] = pd.qcut(df["current_ratio"], q=[0, .33, .67, 1.0], labels=["LOW", "MED", "HIGH"], duplicates="drop")
    df["ocf_to_revenue_sign"] = np.where(df["ocf_to_revenue"] >= 0, "POSITIVE", "NEGATIVE")
    # Xay truoc dang string thuan (khong tron NaN vao np.where cung luc voi
    # string - numpy moi (2.x) khong tu suy ra dtype chung cho str + float NaN
    # va nem DTypePromotionError), roi gan NaN sau qua .loc tren cot dtype
    # object cua pandas (khong di qua np.where nua nen khong bi loi tren).
    df["next_margin_direction"] = np.where(df["margin_up_next_q"] == 1, "UP", "DOWN")
    df.loc[df["margin_up_next_q"].isna(), "next_margin_direction"] = np.nan
    return df


def rule_metrics(df, antecedent_cols, antecedent_vals, consequent_col, consequent_val):
    mask_a = np.ones(len(df), dtype=bool)
    for c, v in zip(antecedent_cols, antecedent_vals):
        mask_a &= (df[c] == v).to_numpy()
    mask_b = (df[consequent_col] == consequent_val).to_numpy()
    n = len(df)
    n_a = int(mask_a.sum())
    support = float((mask_a & mask_b).sum()) / n if n else float("nan")
    confidence = float((mask_a & mask_b).sum()) / n_a if n_a else float("nan")
    p_b = float(mask_b.sum()) / n if n else float("nan")
    lift = confidence / p_b if p_b else float("nan")
    return {"support": support, "confidence": confidence, "lift": lift, "n_antecedent": n_a}


if not panel.empty:
    disc = discretize_for_rules(panel).dropna(
        subset=["leverage_bin", "current_ratio_bin", "ocf_to_revenue_sign", "next_margin_direction"]
    )
    if len(disc) >= 20:
        rule = rule_metrics(disc, ["leverage_bin", "current_ratio_bin"], ["HIGH", "LOW"], "next_margin_direction", "DOWN")
        print("Luat vi du: leverage=HIGH AND current_ratio=LOW -> next_margin=DOWN")
        print(rule)

        candidates = [
            {"leverage": lev, "current_ratio": cur,
             **rule_metrics(disc, ["leverage_bin", "current_ratio_bin"], [lev, cur], "next_margin_direction", "DOWN")}
            for lev in ["LOW", "MED", "HIGH"] for cur in ["LOW", "MED", "HIGH"]
        ]
        rules_df = pd.DataFrame(candidates).sort_values("lift", ascending=False)
        print("\nQuet toan bo to hop leverage x current_ratio -> next_margin=DOWN, sap theo lift:")
        display(rules_df)
    else:
        print(f"Chi {len(disc)} dong du du lieu roi rac hoa - can them de association rule co y nghia.")

Luat vi du: leverage=HIGH AND current_ratio=LOW -> next_margin=DOWN
{'support': 0.10695970695970695, 'confidence': 0.4620253164556962, 'lift': 0.9808157962084376, 'n_antecedent': 316}

Quet toan bo to hop leverage x current_ratio -> next_margin=DOWN, sap theo lift:


,leverage,current_ratio,support,confidence,lift,n_antecedent
1,LOW,MED,0.031502,0.551282,1.170295,78
3,MED,LOW,0.042491,0.495726,1.052359,117
4,MED,MED,0.098901,0.487365,1.034608,277
8,HIGH,HIGH,0.010256,0.482759,1.024830,29
5,MED,HIGH,0.026374,0.467532,0.992507,77
6,HIGH,LOW,0.106960,0.462025,0.980816,316
2,LOW,HIGH,0.112088,0.455357,0.966660,336
0,LOW,LOW,0.008791,0.444444,0.943494,27
7,HIGH,MED,0.033700,0.425926,0.904182,108


## 13. Model-selection scoring

Composite score chỉ là policy hỗ trợ, không thay thế metric gốc.

Official scoring chỉ dùng:

- validation folds;
- `train_policy="full"`;
- không dùng final test;
- model phải thắng persistence;
- nếu chênh MAE dưới 3%, ưu tiên model đơn giản và ổn định hơn.

Không tự động ghi model vào `FINAL_MODEL_SELECTION`; người dùng phải xem kết quả và khóa thủ công.

In [38]:
def composite_regression_score(
    per_model,
    w_mae=0.45,
    w_rmse=0.25,
    w_instability=0.15,
    w_complexity=0.15,
    complexity_rank=None,
):
    frame = per_model.copy()

    frame["mae_norm"] = (
        frame["mae"] - frame["mae"].min()
    ) / (
        frame["mae"].max() - frame["mae"].min() + 1e-12
    )

    frame["rmse_norm"] = (
        frame["rmse"] - frame["rmse"].min()
    ) / (
        frame["rmse"].max() - frame["rmse"].min() + 1e-12
    )

    if complexity_rank is None:
        complexity_rank = {
            "persistence": 0,
            "linear_regression": 1,
            "ridge": 1,
            "lasso": 1,
            "elastic_net": 1,
            "huber": 1,
            "knn": 2,
            "random_forest": 3,
            "extra_trees": 3,
            "svr_rbf": 3,
            "hist_gbm": 4,
            "xgboost": 4,
            "lightgbm": 4,
            "catboost": 4,
            "mlp": 5,
        }

    frame["complexity"] = [
        complexity_rank.get(model, 3)
        for model in frame.index
    ]

    max_complexity = max(
        1,
        frame["complexity"].max(),
    )
    frame["complexity_norm"] = (
        frame["complexity"] / max_complexity
    )

    frame["score"] = (
        w_mae * frame["mae_norm"]
        + w_rmse * frame["rmse_norm"]
        + w_instability
        * frame["instability"].fillna(0)
        + w_complexity
        * frame["complexity_norm"]
    )

    return frame.sort_values("score")


net_margin_result = regression_results.get(
    "net_margin_next_q",
    pd.DataFrame(),
)

if net_margin_result.empty:
    print(
        "Chưa có validation result cho net_margin_next_q."
    )
else:
    validation = net_margin_result[
        (~net_margin_result["is_final_test"])
        & (net_margin_result["train_policy"] == "full")
    ].copy()

    per_model = (
        validation.groupby("model")
        .agg(
            mae=("mae", "mean"),
            rmse=("rmse", "mean"),
            mae_std=("mae", "std"),
            folds=("val_quarter", "nunique"),
        )
    )

    per_model["instability"] = (
        per_model["mae_std"] / per_model["mae"]
    ).replace([np.inf, -np.inf], np.nan).fillna(0)

    scored = composite_regression_score(per_model)
    display(
        scored[
            [
                "folds",
                "mae",
                "rmse",
                "instability",
                "complexity",
                "score",
            ]
        ]
    )

    if "persistence" in per_model.index:
        baseline_mae = per_model.loc[
            "persistence",
            "mae",
        ]

        scored["beats_persistence"] = (
            scored["mae"] < baseline_mae
        )
        scored["within_3pct_of_best"] = (
            scored["mae"]
            <= scored["mae"].min() * 1.03
        )

        eligible = scored[
            scored["beats_persistence"]
            & scored["within_3pct_of_best"]
        ].sort_values(
            ["complexity", "instability", "mae"]
        )

        print(
            "Model phải thắng persistence; "
            "nếu trong 3% best MAE thì ưu tiên đơn giản/ổn định."
        )
        display(
            scored[
                [
                    "mae",
                    "beats_persistence",
                    "within_3pct_of_best",
                    "instability",
                    "complexity",
                    "score",
                ]
            ]
        )

        if not eligible.empty:
            recommended = eligible.index[0]
            print(
                "Gợi ý validation champion:",
                recommended,
            )
            print(
                "Sau khi kiểm tra subgroup/error analysis, "
                "khóa thủ công bằng:"
            )
            print(
                f'FINAL_MODEL_SELECTION["net_margin_next_q"] = "{recommended}"'
            )
        else:
            print(
                "Chưa có model đủ điều kiện thắng persistence."
            )

,folds,mae,rmse,instability,complexity,score
model,,,,,,
persistence,3,0.082886,0.202173,0.273465,0,0.041020
extra_trees,3,0.083036,0.237127,0.202295,3,0.298009
huber,3,0.085551,0.243995,0.171321,1,0.313325
lasso,3,0.087581,0.248889,0.144820,1,0.383172
random_forest,3,0.087406,0.238835,0.145342,3,0.410199
knn,3,0.088199,0.244148,0.116793,2,0.411977
elastic_net,3,0.090167,0.250443,0.140662,1,0.456341
catboost,3,0.086429,0.251833,0.164454,4,0.481484
ridge,3,0.095396,0.250060,0.172933,1,0.595182


Model phải thắng persistence; nếu trong 3% best MAE thì ưu tiên đơn giản/ổn định.


,mae,beats_persistence,within_3pct_of_best,instability,complexity,score
model,,,,,,
persistence,0.082886,False,True,0.273465,0,0.041020
extra_trees,0.083036,False,True,0.202295,3,0.298009
huber,0.085551,False,False,0.171321,1,0.313325
lasso,0.087581,False,False,0.144820,1,0.383172
random_forest,0.087406,False,False,0.145342,3,0.410199
knn,0.088199,False,False,0.116793,2,0.411977
elastic_net,0.090167,False,False,0.140662,1,0.456341
catboost,0.086429,False,False,0.164454,4,0.481484
ridge,0.095396,False,False,0.172933,1,0.595182


Chưa có model đủ điều kiện thắng persistence.


## 14. Out-of-time và out-of-company generalization

- **Out-of-time** là mục tiêu chính: dự báo quý tương lai cho công ty đã có lịch sử.
- **Out-of-company** là cold-start diagnostic: dự báo cho ticker chưa từng thấy.

Out-of-company dùng riêng development data, không dùng final-test quarter. Nó không thay thế walk-forward vì GroupKFold không bảo toàn thứ tự thời gian.

In [39]:
from sklearn.model_selection import GroupKFold


def out_of_company_eval(
    panel,
    task,
    model_name="random_forest",
    n_splits=5,
):
    target = task["target"]
    numeric = task["numeric"]
    categorical = task.get(
        "categorical",
        ["sector", "exchange"],
    )

    if target not in panel.columns:
        return pd.DataFrame()

    # Loại quý feature mới nhất để không chạm final-test window.
    max_feature_quarter = panel.loc[
        panel[target].notna(),
        "quarter_index",
    ].max()

    development = panel[
        (panel[target].notna())
        & (panel["quarter_index"] < max_feature_quarter)
    ].copy()

    n_tickers = development["ticker"].nunique()
    if n_tickers < 4:
        print(
            f"Chỉ có {n_tickers} ticker; "
            "không đủ GroupKFold có ý nghĩa."
        )
        return pd.DataFrame()

    if model_name not in REGRESSION_MODEL_FACTORIES:
        raise KeyError(
            f"Unknown regression model: {model_name}"
        )

    n_splits = min(n_splits, n_tickers)
    splitter = GroupKFold(n_splits=n_splits)
    rows = []

    for fold_id, (train_position, val_position) in enumerate(
        splitter.split(
            development,
            groups=development["ticker"],
        ),
        start=1,
    ):
        train_df = development.iloc[train_position]
        val_df = development.iloc[val_position]

        if train_df.empty or val_df.empty:
            continue

        try:
            preprocess_state = fit_preprocessor(
                train_df,
                numeric,
                categorical,
                scale_numeric=(
                    model_name in SCALED_MODEL_NAMES
                ),
            )
            X_train = transform_matrix(
                train_df,
                preprocess_state,
            )
            X_val = transform_matrix(
                val_df,
                preprocess_state,
            )

            model = REGRESSION_MODEL_FACTORIES[
                model_name
            ]()
            model.fit(
                X_train.to_numpy(dtype=float),
                train_df[target].to_numpy(dtype=float),
            )
            prediction = model.predict(
                X_val.to_numpy(dtype=float)
            )

        except Exception as exc:
            print(
                f"[out-of-company skip] fold={fold_id}: "
                f"{type(exc).__name__}: {exc}"
            )
            continue

        persistence_col = (
            task.get("persistence_col")
            or target.replace("_next_q", "")
        )
        current_values = (
            val_df[persistence_col].to_numpy(dtype=float)
            if persistence_col in val_df.columns
            else None
        )

        rows.append(
            {
                "fold": fold_id,
                "model": model_name,
                "train_tickers": train_df[
                    "ticker"
                ].nunique(),
                "validation_tickers": val_df[
                    "ticker"
                ].nunique(),
                **regression_metrics(
                    val_df[target].to_numpy(dtype=float),
                    prediction,
                    current_values=current_values,
                ),
            }
        )

    return pd.DataFrame(rows)


if not panel.empty:
    net_margin_task = REGRESSION_TASKS[0]
    out_of_company_result = out_of_company_eval(
        panel,
        net_margin_task,
        model_name="random_forest",
    )

    if not out_of_company_result.empty:
        print(
            "Cold-start out-of-company diagnostic "
            "(không thay thế temporal validation):"
        )
        display(out_of_company_result)
        display(
            out_of_company_result.mean(
                numeric_only=True
            ).to_frame("mean_out_of_company")
        )

        out_of_time = regression_results.get(
            "net_margin_next_q",
            pd.DataFrame(),
        )
        if not out_of_time.empty:
            out_of_time_rf = out_of_time[
                (out_of_time["model"] == "random_forest")
                & (~out_of_time["is_final_test"])
                & (out_of_time["train_policy"] == "full")
            ]
            if not out_of_time_rf.empty:
                print(
                    "Out-of-time validation của cùng Random Forest:"
                )
                display(
                    out_of_time_rf[
                        ["mae", "rmse", "r2", "spearman"]
                    ]
                    .mean()
                    .to_frame("mean_out_of_time")
                )

Cold-start out-of-company diagnostic (không thay thế temporal validation):


,fold,model,train_tickers,validation_tickers,n,mae,rmse,r2,spearman,directional_accuracy
0,1,random_forest,141,35,240,0.068047,0.139514,0.134182,0.606259,0.608333
1,2,random_forest,141,35,240,0.099957,0.256776,0.213947,0.645558,0.604167
2,3,random_forest,140,36,242,0.056932,0.105389,0.635672,0.737947,0.668050
3,4,random_forest,141,35,240,0.101076,0.367188,0.177267,0.792659,0.589958
4,5,random_forest,141,35,239,0.130431,0.403674,0.180368,0.628211,0.598326


,mean_out_of_company
fold,3.000000
train_tickers,140.800000
validation_tickers,35.200000
n,240.200000
mae,0.091288
rmse,0.254508
r2,0.268287
spearman,0.682127
directional_accuracy,0.613767


Out-of-time validation của cùng Random Forest:


,mean_out_of_time
mae,0.087406
rmse,0.238835
r2,0.305396
spearman,0.700763


## 15. Export artifacts và final-test report

Cell này:

- gom kết quả từ các task;
- tách validation và final-test rows;
- lưu CSV/JSON vào `artifacts/ml_task_suite/`;
- không tự động thay đổi champion;
- chỉ hiển thị final test khi `RUN_FINAL_TEST=True`.

In [40]:
def _concat_result_dict(result_dict):
    frames = [
        frame
        for frame in result_dict.values()
        if isinstance(frame, pd.DataFrame)
        and not frame.empty
    ]
    return (
        pd.concat(frames, ignore_index=True)
        if frames
        else pd.DataFrame()
    )


all_regression_results = _concat_result_dict(
    regression_results
)
all_classification_results = _concat_result_dict(
    classification_results
)

if not all_regression_results.empty:
    all_regression_results.to_csv(
        ARTIFACT_DIR / "regression_fold_results.csv",
        index=False,
    )

if not all_classification_results.empty:
    all_classification_results.to_csv(
        ARTIFACT_DIR / "classification_fold_results.csv",
        index=False,
    )

if isinstance(ranking_results, pd.DataFrame) and not ranking_results.empty:
    ranking_results.to_csv(
        ARTIFACT_DIR / "ranking_results.csv",
        index=False,
    )

if isinstance(cluster_diagnostics, pd.DataFrame) and not cluster_diagnostics.empty:
    cluster_diagnostics.to_csv(
        ARTIFACT_DIR / "cluster_diagnostics.csv",
        index=False,
    )

if (
    "cluster_transitions" in globals()
    and isinstance(cluster_transitions, pd.DataFrame)
    and not cluster_transitions.empty
):
    cluster_transitions.to_csv(
        ARTIFACT_DIR / "cluster_transitions_fixed_model.csv",
        index=False,
    )

if (
    isinstance(fixed_cluster_assignments, pd.DataFrame)
    and not fixed_cluster_assignments.empty
):
    fixed_cluster_assignments.to_csv(
        ARTIFACT_DIR / "fixed_cluster_assignments.csv",
        index=False,
    )

if (
    isinstance(latest_anomaly_scores, pd.DataFrame)
    and not latest_anomaly_scores.empty
):
    latest_anomaly_scores.to_csv(
        ARTIFACT_DIR / "latest_quarter_anomaly_scores.csv",
        index=False,
    )

if (
    isinstance(synthetic_result, pd.DataFrame)
    and not synthetic_result.empty
):
    synthetic_result.to_csv(
        ARTIFACT_DIR / "synthetic_anomaly_test.csv",
        index=False,
    )

coverage = (
    panel.groupby(
        ["fiscal_year", "quarter_num"],
        as_index=False,
    )
    .agg(
        rows=("ticker", "size"),
        tickers=("ticker", "nunique"),
        cutoff_coverage=(
            "information_cutoff",
            lambda values: float(values.notna().mean()),
        ),
    )
    if not panel.empty
    else pd.DataFrame()
)

if not coverage.empty:
    coverage.to_csv(
        ARTIFACT_DIR / "panel_quarter_coverage.csv",
        index=False,
    )

manifest = {
    "random_state": RANDOM_STATE,
    "run_final_test": RUN_FINAL_TEST,
    "analysis_cutoff_quarter": int(ANALYSIS_CUTOFF_QUARTER),
    "analysis_cutoff_label": quarter_index_to_label(ANALYSIS_CUTOFF_QUARTER),
    "current_analysis_quarter": int(CURRENT_ANALYSIS_QUARTER),
    "current_analysis_label": quarter_index_to_label(CURRENT_ANALYSIS_QUARTER),
    "final_test_target_quarter": int(FINAL_TEST_TARGET_QUARTER),
    "final_test_target_label": quarter_index_to_label(FINAL_TEST_TARGET_QUARTER),
    "complete_quarter_threshold": int(complete_quarter_threshold),
    "complete_quarters": [int(value) for value in complete_quarters],
    "final_model_selection": FINAL_MODEL_SELECTION,
    "require_point_in_time_cutoff": REQUIRE_POINT_IN_TIME_CUTOFF,
    "use_safe_quarterly_macro": USE_SAFE_QUARTERLY_MACRO,
    "flow_value_mode": FLOW_VALUE_MODE,
    "flow_mode_confirmed": FLOW_MODE_CONFIRMED,
    "run_causal": RUN_CAUSAL,
    "panel_rows": int(len(panel)),
    "panel_tickers": (
        int(panel["ticker"].nunique())
        if not panel.empty
        else 0
    ),
    "panel_quarters": (
        sorted(
            int(value)
            for value in panel[
                "quarter_index"
            ].dropna().unique()
        )
        if not panel.empty
        else []
    ),
    "restatement_point_in_time_verified": False,
    "selected_cluster_k": int(SELECTED_CLUSTER_K),
}

with open(
    ARTIFACT_DIR / "experiment_manifest.json",
    "w",
    encoding="utf-8",
) as handle:
    import json

    json.dump(
        manifest,
        handle,
        ensure_ascii=False,
        indent=2,
        default=str,
    )

print("Artifacts written to:", ARTIFACT_DIR)
display(coverage)

if RUN_FINAL_TEST:
    final_frames = []

    if not all_regression_results.empty:
        final_frames.append(
            all_regression_results[
                all_regression_results["is_final_test"]
            ].copy()
        )

    if not all_classification_results.empty:
        final_frames.append(
            all_classification_results[
                all_classification_results["is_final_test"]
            ].copy()
        )

    final_report = (
        pd.concat(final_frames, ignore_index=True)
        if final_frames
        else pd.DataFrame()
    )

    print(
        "FINAL TEST REPORT — chỉ hợp lệ nếu "
        "FINAL_MODEL_SELECTION đã được khóa trước khi chạy."
    )
    display(final_report)

    if not final_report.empty:
        final_report.to_csv(
            ARTIFACT_DIR / "final_test_report.csv",
            index=False,
        )
else:
    print(
        "Final test vẫn đang khóa. "
        "Đây là trạng thái đúng trong giai đoạn model development."
    )

Artifacts written to: /Users/mac/Desktop/Project/Thesis_Project/artifacts/ml_task_suite


,fiscal_year,quarter_num,rows,tickers,cutoff_coverage
0,2024,1,173,173,1.0
1,2024,2,174,174,1.0
2,2024,3,174,174,1.0
3,2024,4,176,176,1.0
4,2025,1,173,173,1.0
5,2025,2,174,174,1.0
6,2025,3,176,176,1.0
7,2025,4,171,171,1.0
8,2026,1,170,170,1.0


Final test vẫn đang khóa. Đây là trạng thái đúng trong giai đoạn model development.


## 16. Xuất prediction có cấu trúc cho RAG

RAG **không nên đọc notebook stdout**. Sau khi model được khóa và final test đã được
chấp nhận, notebook sẽ:

1. train lại champion trên toàn bộ label đã quan sát đến `CURRENT_ANALYSIS_QUARTER`;
2. dự đoán quý kế tiếp cho từng ticker;
3. lưu model bundle bằng `joblib`;
4. xuất:
   - `rag_predictions.jsonl`: một prediction record cho mỗi ticker/task;
   - `rag_predictions.csv`: bản phẳng để kiểm tra hoặc query SQL;
   - `rag_model_cards.json`: validation metrics, feature list và model version;
   - `rag_context_preview.md`: văn bản dễ chunk/embed;
5. tùy chọn ghi bản phẳng vào PostgreSQL khi `WRITE_RAG_OUTPUT_TO_DB=True`.

Mỗi record nói rõ đây là **predictive output, không phải causal evidence**. Khi trả lời,
router nên lấy:

- prediction/model card từ structured store;
- BCTC, disclosure và news evidence từ RAG/web;
- sau đó mới cho LLM tổng hợp.

Để bật:

```python
FINAL_MODEL_SELECTION = {
    "net_margin_next_q": "huber",
    "margin_up_next_q": "random_forest",
    "margin_negative_next_q": "random_forest",
    "leverage_next_q": "persistence",
}
MODELS_LOCKED_FOR_RAG = True
EXPORT_RAG_OUTPUTS = True
```

Không bật `MODELS_LOCKED_FOR_RAG` trước khi rerun validation với complete-quarter gate
và chấp nhận final test.

In [41]:
import json
import joblib
from datetime import datetime, timezone


REGRESSION_TASK_MAP = {
    task["name"]: task
    for task in REGRESSION_TASKS
}
CLASSIFICATION_TASK_MAP = {
    task["name"]: task
    for task in CLASSIFICATION_TASKS
}


def _json_safe(value):
    if value is None:
        return None
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return None if not np.isfinite(value) else float(value)
    if isinstance(value, (pd.Timestamp, datetime)):
        return value.isoformat()
    if pd.isna(value):
        return None
    return value


def _task_definition(task_name):
    if task_name in REGRESSION_TASK_MAP:
        return "regression", REGRESSION_TASK_MAP[task_name]
    if task_name in CLASSIFICATION_TASK_MAP:
        return "classification", CLASSIFICATION_TASK_MAP[task_name]
    raise KeyError(f"Không tìm thấy task: {task_name}")


def _validation_model_card_metrics(
    task_name,
    task_type,
    model_name,
):
    result_map = (
        regression_results
        if task_type == "regression"
        else classification_results
    )
    frame = result_map.get(task_name, pd.DataFrame())

    if frame.empty:
        return {}

    validation = frame[
        (~frame["is_final_test"])
        & (frame["model"] == model_name)
        & (frame["train_policy"] == "full")
    ].copy()

    if validation.empty:
        return {}

    metric_cols = (
        [
            "mae",
            "rmse",
            "r2",
            "spearman",
            "directional_accuracy",
        ]
        if task_type == "regression"
        else [
            "balanced_accuracy",
            "f1",
            "roc_auc",
            "pr_auc",
            "brier",
            "prevalence",
        ]
    )

    metrics = {}
    for col in metric_cols:
        if col in validation.columns:
            metrics[f"{col}_mean"] = _json_safe(
                validation[col].mean()
            )
            metrics[f"{col}_std"] = _json_safe(
                validation[col].std()
            )

    metrics["validation_folds"] = int(
        validation["val_quarter"].nunique()
    )
    return metrics


def _global_model_attribution(
    model,
    preprocess_state,
    top_k=12,
):
    feature_names = list(
        preprocess_state.get(
            "output_columns",
            [],
        )
    )

    values = None
    attribution_type = None

    if hasattr(model, "feature_importances_"):
        values = np.asarray(
            model.feature_importances_
        ).reshape(-1)
        attribution_type = "global_feature_importance"
    elif hasattr(model, "coef_"):
        values = np.asarray(
            model.coef_
        )
        if values.ndim > 1:
            values = np.mean(
                np.abs(values),
                axis=0,
            )
        else:
            values = np.abs(values)
        values = values.reshape(-1)
        attribution_type = "absolute_coefficient"

    if (
        values is None
        or not feature_names
        or len(values) != len(feature_names)
    ):
        return {
            "type": "unavailable",
            "items": [],
            "warning": (
                "Không có local causal explanation. "
                "Feature importance/coefficient nếu có chỉ là model attribution."
            ),
        }

    order = np.argsort(-np.abs(values))[:top_k]
    items = [
        {
            "feature": str(feature_names[index]),
            "value": _json_safe(values[index]),
        }
        for index in order
    ]

    return {
        "type": attribution_type,
        "items": items,
        "warning": (
            "Global model attribution; không phải causal effect "
            "và không phải local explanation cho riêng ticker."
        ),
    }


def _fit_locked_task_for_future(
    frame,
    task_name,
    model_name,
):
    task_type, task = _task_definition(task_name)

    if (
        task.get("requires_flow_mode_confirmed", False)
        and not FLOW_MODE_CONFIRMED
        and not ALLOW_UNCONFIRMED_FLOW_TASKS
    ):
        raise RuntimeError(
            f"{task_name} phụ thuộc flow convention nhưng "
            "FLOW_MODE_CONFIRMED=False."
        )

    target = task["target"]
    numeric = task["numeric"]
    categorical = task.get(
        "categorical",
        ["sector", "exchange"],
    )

    train = frame[
        frame[target].notna()
        & frame["target_quarter_index"].notna()
        & (
            frame["target_quarter_index"]
            <= CURRENT_ANALYSIS_QUARTER
        )
    ].copy()

    score = frame[
        frame["quarter_index"]
        == CURRENT_ANALYSIS_QUARTER
    ].copy()

    if train.empty:
        raise RuntimeError(
            f"Không có labeled training rows cho {task_name}."
        )
    if score.empty:
        raise RuntimeError(
            "Không có feature rows tại CURRENT_ANALYSIS_QUARTER."
        )

    model = None
    preprocess_state = None
    prediction = None
    probability = None
    threshold = None

    if model_name == "persistence":
        if task_type != "regression":
            raise ValueError(
                "Persistence hiện chỉ hỗ trợ regression."
            )
        persistence_col = (
            task.get("persistence_col")
            or target.replace("_next_q", "")
        )
        if persistence_col not in score.columns:
            raise KeyError(
                f"Thiếu persistence column: {persistence_col}"
            )
        prediction = score[
            persistence_col
        ].to_numpy(dtype=float)
        attribution = {
            "type": "persistence_rule",
            "items": [
                {
                    "feature": persistence_col,
                    "value": 1.0,
                }
            ],
            "warning": (
                "Prediction quý sau bằng giá trị quý hiện tại; "
                "không phải causal effect."
            ),
        }
    else:
        factories = (
            REGRESSION_MODEL_FACTORIES
            if task_type == "regression"
            else CLASSIFICATION_MODEL_FACTORIES
        )
        factory = factories.get(model_name)
        if factory is None:
            raise KeyError(
                f"Model {model_name} không tồn tại cho {task_type}."
            )

        preprocess_state = fit_preprocessor(
            train,
            numeric,
            categorical,
            scale_numeric=(
                model_name in SCALED_MODEL_NAMES
            ),
        )
        X_train = transform_matrix(
            train,
            preprocess_state,
        )
        X_score = transform_matrix(
            score,
            preprocess_state,
        )

        y_train = train[
            target
        ].to_numpy(dtype=float)

        model = factory()

        if (
            task_type == "classification"
            and model_name == "xgboost"
        ):
            positives = max(
                1,
                int(np.sum(y_train == 1)),
            )
            negatives = max(
                1,
                int(np.sum(y_train == 0)),
            )
            model.set_params(
                scale_pos_weight=(
                    negatives / positives
                )
            )

        model.fit(
            X_train.to_numpy(dtype=float),
            y_train,
        )

        if task_type == "regression":
            prediction = model.predict(
                X_score.to_numpy(dtype=float)
            )
        else:
            if hasattr(model, "predict_proba"):
                probability = model.predict_proba(
                    X_score.to_numpy(dtype=float)
                )[:, 1]
            elif hasattr(model, "decision_function"):
                decision = model.decision_function(
                    X_score.to_numpy(dtype=float)
                )
                decision = np.clip(
                    decision,
                    -30,
                    30,
                )
                probability = 1.0 / (
                    1.0 + np.exp(-decision)
                )
            else:
                probability = model.predict(
                    X_score.to_numpy(dtype=float)
                ).astype(float)

            threshold = float(
                CLASSIFICATION_THRESHOLDS.get(
                    task_name,
                    0.50,
                )
            )

        attribution = _global_model_attribution(
            model,
            preprocess_state,
        )

    run_timestamp = datetime.now(
        timezone.utc
    ).strftime("%Y%m%dT%H%M%SZ")
    model_version = (
        f"{RAG_MODEL_VERSION_PREFIX}:"
        f"{task_name}:{model_name}:"
        f"train_through_"
        f"{quarter_index_to_label(CURRENT_ANALYSIS_QUARTER)}:"
        f"{run_timestamp}"
    )

    bundle = {
        "task_name": task_name,
        "task_type": task_type,
        "target": target,
        "model_name": model_name,
        "model_version": model_version,
        "trained_through_target_quarter": int(
            CURRENT_ANALYSIS_QUARTER
        ),
        "trained_through_target_label": (
            quarter_index_to_label(
                CURRENT_ANALYSIS_QUARTER
            )
        ),
        "future_feature_quarter": int(
            CURRENT_ANALYSIS_QUARTER
        ),
        "future_target_quarter": int(
            CURRENT_ANALYSIS_QUARTER + 1
        ),
        "numeric_features": numeric,
        "categorical_features": categorical,
        "preprocess_state": preprocess_state,
        "model": model,
        "attribution": attribution,
    }

    model_path = (
        ARTIFACT_DIR
        / f"locked_model__{task_name}__{model_name}.joblib"
    )
    joblib.dump(bundle, model_path)

    validation_metrics = (
        _validation_model_card_metrics(
            task_name,
            task_type,
            model_name,
        )
    )

    model_card = {
        "record_type": "ml_model_card",
        "task": task_name,
        "task_type": task_type,
        "target": target,
        "model_name": model_name,
        "model_version": model_version,
        "training_rows": int(len(train)),
        "training_tickers": int(
            train["ticker"].nunique()
        ),
        "training_target_quarter_min": (
            quarter_index_to_label(
                int(
                    train[
                        "target_quarter_index"
                    ].min()
                )
            )
        ),
        "training_target_quarter_max": (
            quarter_index_to_label(
                int(
                    train[
                        "target_quarter_index"
                    ].max()
                )
            )
        ),
        "validation_metrics": validation_metrics,
        "classification_threshold": threshold,
        "global_attribution": attribution,
        "flow_mode_confirmed": bool(
            FLOW_MODE_CONFIRMED
        ),
        "point_in_time_cutoff_required": bool(
            REQUIRE_POINT_IN_TIME_CUTOFF
        ),
        "claim_scope": "predictive_not_causal",
        "model_artifact": str(model_path),
    }

    records = []

    for position, (_, row) in enumerate(
        score.iterrows()
    ):
        input_snapshot = {}
        for col in numeric + categorical:
            if col in score.columns:
                input_snapshot[col] = _json_safe(
                    row[col]
                )

        if task_type == "regression":
            predicted_value = _json_safe(
                prediction[position]
            )
            predicted_probability = None
            predicted_class = None
        else:
            predicted_value = None
            predicted_probability = _json_safe(
                probability[position]
            )
            predicted_class = int(
                probability[position]
                >= threshold
            )

        cutoff = _json_safe(
            row.get("information_cutoff")
        )

        record = {
            "record_type": "ml_prediction",
            "task": task_name,
            "task_type": task_type,
            "ticker": str(row["ticker"]),
            "sector": _json_safe(
                row.get("sector")
            ),
            "exchange": _json_safe(
                row.get("exchange")
            ),
            "as_of_quarter_index": int(
                CURRENT_ANALYSIS_QUARTER
            ),
            "as_of_quarter": (
                quarter_index_to_label(
                    CURRENT_ANALYSIS_QUARTER
                )
            ),
            "target_quarter_index": int(
                CURRENT_ANALYSIS_QUARTER + 1
            ),
            "target_quarter": (
                quarter_index_to_label(
                    CURRENT_ANALYSIS_QUARTER + 1
                )
            ),
            "horizon_quarters": 1,
            "target": target,
            "model_name": model_name,
            "model_version": model_version,
            "prediction": predicted_value,
            "probability": predicted_probability,
            "predicted_class": predicted_class,
            "classification_threshold": threshold,
            "information_cutoff": cutoff,
            "validation_metrics": validation_metrics,
            "input_snapshot": input_snapshot,
            "global_attribution": attribution,
            "uncertainty_status": (
                "not_calibrated"
            ),
            "claim_scope": (
                "predictive_not_causal"
            ),
            "rag_instruction": (
                "Use this structured prediction as model evidence. "
                "Do not describe it as a causal effect. "
                "Retrieve financial-disclosure/news evidence separately "
                "before explaining why the prediction may be plausible."
            ),
        }

        if task_type == "regression":
            result_text = (
                f"prediction={predicted_value}"
            )
        else:
            result_text = (
                f"probability={predicted_probability}; "
                f"class={predicted_class}; "
                f"threshold={threshold}"
            )

        record["rag_text"] = (
            f"Structured ML forecast for {record['ticker']}. "
            f"Task={task_name}. "
            f"As of {record['as_of_quarter']}, "
            f"target quarter={record['target_quarter']}, "
            f"{result_text}. "
            f"Model={model_name}, version={model_version}. "
            "This is predictive, not causal. "
            "Validation metrics are attached in the structured record."
        )
        records.append(record)

    return records, model_card, bundle


rag_prediction_records = []
rag_model_cards = []
locked_model_bundles = {}

if EXPORT_RAG_OUTPUTS:
    if not MODELS_LOCKED_FOR_RAG:
        raise RuntimeError(
            "EXPORT_RAG_OUTPUTS=True nhưng MODELS_LOCKED_FOR_RAG=False. "
            "Chỉ bật sau khi final test được chấp nhận."
        )
    if not FINAL_MODEL_SELECTION:
        raise RuntimeError(
            "Chưa khai báo FINAL_MODEL_SELECTION."
        )

    for task_name, model_name in (
        FINAL_MODEL_SELECTION.items()
    ):
        records, model_card, bundle = (
            _fit_locked_task_for_future(
                panel,
                task_name,
                model_name,
            )
        )
        rag_prediction_records.extend(records)
        rag_model_cards.append(model_card)
        locked_model_bundles[task_name] = bundle

    rag_jsonl_path = (
        ARTIFACT_DIR / "rag_predictions.jsonl"
    )
    with open(
        rag_jsonl_path,
        "w",
        encoding="utf-8",
    ) as handle:
        for record in rag_prediction_records:
            handle.write(
                json.dumps(
                    record,
                    ensure_ascii=False,
                    default=str,
                )
                + "\n"
            )

    rag_cards_path = (
        ARTIFACT_DIR / "rag_model_cards.json"
    )
    with open(
        rag_cards_path,
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            rag_model_cards,
            handle,
            ensure_ascii=False,
            indent=2,
            default=str,
        )

    rag_flat_rows = []
    for record in rag_prediction_records:
        rag_flat_rows.append(
            {
                "task": record["task"],
                "task_type": record["task_type"],
                "ticker": record["ticker"],
                "sector": record["sector"],
                "exchange": record["exchange"],
                "as_of_quarter": record["as_of_quarter"],
                "target_quarter": record["target_quarter"],
                "target": record["target"],
                "model_name": record["model_name"],
                "model_version": record["model_version"],
                "prediction": record["prediction"],
                "probability": record["probability"],
                "predicted_class": record["predicted_class"],
                "classification_threshold": record[
                    "classification_threshold"
                ],
                "information_cutoff": record[
                    "information_cutoff"
                ],
                "uncertainty_status": record[
                    "uncertainty_status"
                ],
                "claim_scope": record[
                    "claim_scope"
                ],
                "validation_metrics_json": json.dumps(
                    record["validation_metrics"],
                    ensure_ascii=False,
                ),
                "input_snapshot_json": json.dumps(
                    record["input_snapshot"],
                    ensure_ascii=False,
                ),
                "rag_text": record["rag_text"],
            }
        )

    rag_predictions_df = pd.DataFrame(
        rag_flat_rows
    )
    rag_csv_path = (
        ARTIFACT_DIR / "rag_predictions.csv"
    )
    rag_predictions_df.to_csv(
        rag_csv_path,
        index=False,
    )

    rag_markdown_path = (
        ARTIFACT_DIR / "rag_context_preview.md"
    )
    with open(
        rag_markdown_path,
        "w",
        encoding="utf-8",
    ) as handle:
        for record in rag_prediction_records:
            handle.write(
                "## "
                f"{record['ticker']} — "
                f"{record['task']}\n\n"
            )
            handle.write(
                record["rag_text"]
                + "\n\n"
            )
            handle.write(
                "- Claim scope: "
                f"{record['claim_scope']}\n"
            )
            handle.write(
                "- Uncertainty: "
                f"{record['uncertainty_status']}\n"
            )
            handle.write(
                "- Information cutoff: "
                f"{record['information_cutoff']}\n\n"
            )

    if WRITE_RAG_OUTPUT_TO_DB:
        rag_predictions_df.to_sql(
            "ml_prediction_outputs",
            engine,
            if_exists="append",
            index=False,
            method="multi",
            chunksize=500,
        )

    print(
        "RAG outputs written:",
        rag_jsonl_path,
        rag_csv_path,
        rag_cards_path,
        rag_markdown_path,
        sep="\n- ",
    )
    display(
        rag_predictions_df.head(20)
    )
else:
    print(
        "RAG export đang khóa. Sau khi chốt model, đặt "
        "MODELS_LOCKED_FOR_RAG=True và EXPORT_RAG_OUTPUTS=True."
    )

RAG export đang khóa. Sau khi chốt model, đặt MODELS_LOCKED_FOR_RAG=True và EXPORT_RAG_OUTPUTS=True.


## 17. Tổng kết sử dụng notebook

### Trạng thái phù hợp để chạy exploratory

Notebook hiện đã sửa các vấn đề chính:

- macro annual không còn đi vào panel quý;
- có `information_cutoff`;
- duplicate không còn được chọn bằng `first`;
- preprocessing theo model family;
- all-null feature xử lý theo train fold;
- directional accuracy đúng định nghĩa;
- final test có khóa;
- ranking group theo quý;
- causal outcome dùng quý sau;
- cluster transition dùng fixed model;
- anomaly fit lịch sử và score hiện tại.

### Các điều kiện vẫn cần xác minh từ dữ liệu thật

1. `FLOW_VALUE_MODE`: Q1..Q4 là standalone hay YTD cumulative.
2. Line-item version có liên kết chính xác với disclosure version hay chưa.
3. `information_cutoff` exact/fallback coverage.
4. Số sample thật sau khi yêu cầu point-in-time.
5. Class imbalance và số positive cases của từng classification task.
6. Macro quarterly có `release_date` đủ tốt hay chưa.

### Cách mở final test

Sau khi chốt model trên validation:

```python
RUN_FINAL_TEST = True
FINAL_MODEL_SELECTION = {
    "net_margin_next_q": "elastic_net",
    "revenue_growth_next_q": "random_forest",
    "ocf_next_q": "huber",
    "leverage_next_q": "xgboost",
    "margin_up_next_q": "logistic",
}
```

Sau đó **Restart Kernel → Run All** đúng một lần. Không sửa model theo final-test result.

### Khi dữ liệu mở rộng

Khi có 10–15 năm dữ liệu quý:

- thêm lag 4/8/12;
- rolling 4/8/12;
- expanding vs rolling windows;
- forecast revision ledger;
- model drift monitoring;
- sector VAR/SVAR;
- PCMCI+;
- sequence models chỉ khi timestep và sample size đủ.

Structured RAG output được xuất dưới dạng JSONL/CSV/model cards; notebook stdout không phải retrieval interface.